In [ ]:
import nbformat
import os

NOTEBOOK_PATH = "/content/drive/MyDrive/Colab Notebooks/Copy of Final-Liver_Disease_Prediction_Mortality_Prediction.ipynb"

# Load notebook
with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

# ============================================================
# REMOVE WIDGET METADATA
# ============================================================

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

# ============================================================
# REMOVE PROBLEMATIC CELL OUTPUT METADATA
# ============================================================

for cell in nb.cells:

    # Clear outputs completely
    if "outputs" in cell:
        cell["outputs"] = []

    # Reset execution count
    if "execution_count" in cell:
        cell["execution_count"] = None

    # Remove metadata from each cell
    if "metadata" in cell:
        cell["metadata"] = {}

# ============================================================
# SAVE CLEAN NOTEBOOK
# ============================================================

CLEAN_PATH = "/content/drive/MyDrive/Colab Notebooks/CLEAN_Final_Liver_d.ipynb"

with open(CLEAN_PATH, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("✅ Clean notebook saved successfully")
print(f"Saved at: {CLEAN_PATH}")

In [ ]:
import os

# Set your Kaggle API token directly
os.environ['KAGGLE_TOKEN'] = 'KGAT_1910776b4329692688fc4d6f20d58324'  # ← replace after regenerating

# Create kaggle directory and credentials file
os.makedirs('/root/.kaggle', exist_ok=True)

# Write kaggle.json using the token
# Your Kaggle username is needed too — find it at kaggle.com/your-profile
KAGGLE_USERNAME = 'anshikapal05'  # ← replace with YOUR kaggle username
KAGGLE_KEY = 'KGAT_1910776b4329692688fc4d6f20d58324'  # ← replace with your new token

import json
kaggle_creds = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Kaggle credentials set!")

In [ ]:
!pip install kaggle -q

!kaggle datasets download -d msafi04/predict-mortality-of-icu-patients-physionet --path /content/
print("✅ Download complete!")

In [ ]:
import zipfile

with zipfile.ZipFile('/content/predict-mortality-of-icu-patients-physionet.zip', 'r') as z:
    z.extractall('/content/physionet_data')

# Check what's inside
import os
for item in os.listdir('/content/physionet_data'):
    print(item)

In [ ]:
import os, pandas as pd

# Corrected path to the actual data files
set_a_path = '/content/physionet_data/set-a/set-a/'
all_long = []

files = [f for f in os.listdir(set_a_path) if f.endswith('.txt')]
print(f"Total patient files: {len(files)}")

for filename in files:
    record_id = int(filename.replace('.txt', ''))
    try:
        df = pd.read_csv(os.path.join(set_a_path, filename))
        df['RecordID'] = record_id
        all_long.append(df)
    except:
        pass

long_df = pd.concat(all_long, ignore_index=True)
print(f"Total rows: {long_df.shape}")
print(long_df.head(8))

In [ ]:
# Total patients
total_patients = long_df['RecordID'].nunique()
print(f"Total patients: {total_patients}")

# Count how many patients have EACH parameter at least once
feature_coverage = (
    long_df.groupby('Parameter')['RecordID']
    .nunique()
    .reset_index()
    .rename(columns={'RecordID': 'patient_count'})
)
feature_coverage['coverage_pct'] = (feature_coverage['patient_count'] / total_patients * 100).round(1)
feature_coverage = feature_coverage.sort_values('coverage_pct', ascending=False)

print("\n=== ALL FEATURES & COVERAGE ===")
print(feature_coverage.to_string(index=False))

STEP 2: FEATURE ENGINEERING

In [ ]:
# All 37 parameters in this dataset
# Liver-relevant ones (clinically proven markers of liver disease/failure)
LIVER_FEATURES = [
    'Bilirubin',    # Liver jaundice marker — KEY
    'ALT',          # Liver enzyme (alanine aminotransferase) — KEY
    'AST',          # Liver enzyme (aspartate aminotransferase) — KEY
    'ALP',          # Alkaline phosphatase — liver/bone
    'Albumin',      # Liver makes albumin — KEY for liver failure
    'HCO3',         # Acid-base balance (liver affects this)
    'Creatinine',   # Kidney — often fails alongside liver
    'BUN',          # Blood urea nitrogen — liver makes urea
    'Glucose',      # Liver regulates blood sugar
    'Platelets',    # Low in liver disease (portal hypertension)
    'INR',          # Clotting — liver makes clotting factors (if present)
    'PT',           # Prothrombin time — liver function
    'Na',           # Sodium — liver failure causes hyponatremia
    'K',            # Potassium
]

# Support features (vital signs — high coverage, good predictors)
SUPPORT_FEATURES = [
    'HR', 'NISysABP', 'NIDiasABP', 'NIMAP',
    'RespRate', 'Temp', 'GCS', 'Urine',
    # Demographics (always present at time=00:00)
    'Age', 'Gender', 'Weight', 'Height', 'ICUType'
]

# Check which liver features are actually available and their coverage
liver_coverage = feature_coverage[feature_coverage['Parameter'].isin(LIVER_FEATURES)]
print("=== LIVER FEATURE COVERAGE ===")
print(liver_coverage.to_string(index=False))

# Keep only liver features with >20% coverage (adjust threshold as needed)
THRESHOLD = 20.0
available_liver = liver_coverage[liver_coverage['coverage_pct'] >= THRESHOLD]['Parameter'].tolist()
print(f"\n✅ Liver features with >{THRESHOLD}% coverage: {available_liver}")

In [ ]:
ALL_FEATURES = available_liver + SUPPORT_FEATURES

# Filter long_df to only parameters we care about
filtered_long = long_df[long_df['Parameter'].isin(ALL_FEATURES)]

# Pivot: 1 row per patient, mean of each parameter
patients_wide = (
    filtered_long
    .groupby(['RecordID', 'Parameter'])['Value']
    .mean()
    .unstack('Parameter')
    .reset_index()
)

print(f"Wide dataset shape: {patients_wide.shape}")
print(f"Columns: {list(patients_wide.columns)}")

In [ ]:
# Count how many liver features each patient has (non-NaN)
patients_wide['liver_feature_count'] = patients_wide[available_liver].notna().sum(axis=1)

print("Distribution of liver feature count per patient:")
print(patients_wide['liver_feature_count'].value_counts().sort_index())

# Keep patients with AT LEAST 3 liver features recorded
MIN_LIVER_FEATURES = 3
liver_patients = patients_wide[patients_wide['liver_feature_count'] >= MIN_LIVER_FEATURES].copy()

print(f"\n✅ Patients with ≥{MIN_LIVER_FEATURES} liver features: {len(liver_patients)} / {len(patients_wide)}")
print(f"   Kept: {len(liver_patients)/len(patients_wide)*100:.1f}% of all patients")

In [ ]:
outcomes = pd.read_csv('/content/physionet_data/Outcomes-a.txt')
print("Outcomes columns:", list(outcomes.columns))
# Columns: RecordID, SAPS-I, SOFA, Length_of_stay, Survival, In-hospital_death

final_df = liver_patients.merge(
    outcomes[['RecordID', 'In-hospital_death', 'SAPS-I', 'SOFA']],
    on='RecordID', how='inner'
)

print(f"\n✅ Final merged dataset: {final_df.shape}")
print(f"Death rate: {final_df['In-hospital_death'].mean()*100:.1f}%")
print(f"Survived: {(final_df['In-hospital_death']==0).sum()}")
print(f"Died:     {(final_df['In-hospital_death']==1).sum()}")

In [ ]:
# Re-run your pipeline but save BEFORE imputation
# Add this line right before the imputer step:
final_df_raw = final_df.copy()   # save unimputed version

# Core liver enzyme labs — the real diagnostic markers
CORE_LIVER_LABS = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin']

# Count how many of the 5 core labs each patient actually has (not NaN)
final_df_raw['core_labs_present'] = final_df_raw[CORE_LIVER_LABS].notna().sum(axis=1)

print("Core liver labs present per patient:")
print(final_df_raw['core_labs_present'].value_counts().sort_index())
print(f"\nTotal patients: {len(final_df_raw)}")

In [ ]:
# RECOMMENDED: keep patients with at least 2 of the 5 core liver labs
liver_filtered = final_df_raw[final_df_raw['core_labs_present'] >= 2].copy()

print(f"Before filter: {len(final_df_raw)} patients")
print(f"After filter:  {len(liver_filtered)} patients")
print(f"Removed:       {len(final_df_raw) - len(liver_filtered)} patients")
print(f"\nMortality rate preserved: {liver_filtered['In-hospital_death'].mean()*100:.1f}%")
print(f"  Survived: {(liver_filtered['In-hospital_death']==0).sum()}")
print(f"  Died:     {(liver_filtered['In-hospital_death']==1).sum()}")

In [ ]:
# Keep only the most useful features — drop redundant vitals
# Reasoning:
#   NIDiasABP + NISysABP + NIMAP are all blood pressure → keep only NISysABP
#   HR + RespRate + Temp are generic → keep HR only (highest coverage)
#   Drop ICUType, Height, Weight (not liver-relevant)
#   Drop SAPS-I / SOFA (derived scores, causes data leakage)

FINAL_FEATURES = [
    # Core liver labs
    'Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin',
    # Supporting labs (high coverage, liver-relevant)
    'Creatinine', 'BUN', 'Na', 'K', 'HCO3',
    'Platelets', 'Glucose',
    # Minimal vitals
    'HR', 'NISysABP',
    # Demographics
    'Age', 'Gender',
    # Target
    'In-hospital_death', 'RecordID'
]

# Keep only columns that exist
cols_to_keep = [c for c in FINAL_FEATURES if c in liver_filtered.columns]
liver_slim = liver_filtered[cols_to_keep].copy()

print(f"Columns reduced: {final_df_raw.shape[1]} → {liver_slim.shape[1]}")
print(f"Patients:        {len(liver_slim)}")
print(f"\nFinal columns: {[c for c in cols_to_keep if c not in ['In-hospital_death','RecordID']]}")

In [ ]:
from sklearn.impute import SimpleImputer

feature_cols = [c for c in liver_slim.columns if c not in ['RecordID', 'In-hospital_death']]

# Check missing before imputation
print("Missing % per feature:")
missing = liver_slim[feature_cols].isnull().mean().sort_values(ascending=False) * 100
print(missing[missing > 0].round(1))

# Median impute
imputer = SimpleImputer(strategy='median')
liver_slim[feature_cols] = imputer.fit_transform(liver_slim[feature_cols])

print(f"\nAny NaN remaining: {liver_slim[feature_cols].isnull().any().any()}")

In [ ]:
liver_slim.to_csv('/content/liver_final.csv', index=False)

print("="*45)
print("FINAL LIVER DATASET")
print("="*45)
print(f"Patients:  {len(liver_slim)}")
print(f"Features:  {len(feature_cols)}")
print(f"Mortality: {liver_slim['In-hospital_death'].mean()*100:.1f}%")
print(f"Died:      {liver_slim['In-hospital_death'].sum()}")
print(f"Survived:  {(liver_slim['In-hospital_death']==0).sum()}")
print(f"\nSaved to: liver_final.csv")
liver_slim.head()

STEP3: EDA

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install xgboost lightgbm catboost shap pgmpy -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False
})

print("✅ All libraries imported successfully")
print(f"Dataset shape: {liver_slim.shape}")
liver_slim.head()

In [ ]:
# ============================================================
# CELL 2: Basic dataset overview
# ============================================================
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Shape:    {liver_slim.shape}")
print(f"Patients: {liver_slim.shape[0]}")
print(f"Features: {liver_slim.shape[1]}")
print(f"\nTarget Distribution:")
print(liver_slim['In-hospital_death'].value_counts())
print(f"\nMortality Rate: {liver_slim['In-hospital_death'].mean()*100:.1f}%")

print("\n--- Data Types ---")
print(liver_slim.dtypes)

print("\n--- Descriptive Statistics ---")
liver_slim.describe().T.round(2)

In [ ]:
# ============================================================
# CREATE LIVER DISEASE LABEL (Stage 1 target)
# Clinical thresholds for liver disease detection
# ============================================================

def create_liver_disease_label(df):
    """
    Liver disease = patient has abnormal values in >= 2 liver biomarkers
    Clinical thresholds (adult normal ranges):
      ALT     > 56 U/L
      AST     > 40 U/L
      ALP     > 120 U/L
      Bilirubin > 1.2 mg/dL
      Albumin < 3.5 g/dL  (low = liver dysfunction)
    """
    conditions = pd.DataFrame()
    conditions['ALT_high']       = df['ALT']       > 56
    conditions['AST_high']       = df['AST']       > 40
    conditions['ALP_high']       = df['ALP']       > 120
    conditions['Bili_high']      = df['Bilirubin'] > 1.2
    conditions['Albumin_low']    = df['Albumin']   < 3.5

    # Patient flagged as liver disease if >= 2 markers abnormal
    df['liver_disease'] = (conditions.sum(axis=1) >= 2).astype(int)
    return df

liver_slim = create_liver_disease_label(liver_slim)

print("Liver Disease Label Distribution:")
print(liver_slim['liver_disease'].value_counts())
print(f"\nLiver Disease Rate: {liver_slim['liver_disease'].mean()*100:.1f}%")
print(f"Mortality Rate:     {liver_slim['In-hospital_death'].mean()*100:.1f}%")

# Cross-tabulation: liver disease vs mortality
print("\nCross-tab — Liver Disease vs Mortality:")
ct = pd.crosstab(liver_slim['liver_disease'], liver_slim['In-hospital_death'],
                 rownames=['Liver Disease'], colnames=['In-hospital Death'])
ct.index = ['No Liver Disease', 'Liver Disease']
ct.columns = ['Survived', 'Died']
print(ct)

In [ ]:
# ============================================================
# EDA 1: CLASS DISTRIBUTION
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Class Distribution Analysis', fontsize=14, fontweight='bold', y=1.02)

# 1a. Liver Disease distribution
ax = axes[0]
counts = liver_slim['liver_disease'].value_counts()
bars = ax.bar(['No Liver Disease', 'Liver Disease'], counts.values,
              color=['#4CAF50', '#F44336'], width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val}\n({val/len(liver_slim)*100:.1f}%)',
            ha='center', va='bottom', fontweight='bold')
ax.set_title('Liver Disease Prevalence')
ax.set_ylabel('Patient Count')
ax.set_ylim(0, max(counts.values) * 1.2)

# 1b. Mortality distribution
ax = axes[1]
counts_m = liver_slim['In-hospital_death'].value_counts()
bars2 = ax.bar(['Survived', 'Died'], counts_m.values,
               color=['#2196F3', '#FF5722'], width=0.5, edgecolor='white')
for bar, val in zip(bars2, counts_m.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val}\n({val/len(liver_slim)*100:.1f}%)',
            ha='center', va='bottom', fontweight='bold')
ax.set_title('ICU Mortality Distribution')
ax.set_ylabel('Patient Count')
ax.set_ylim(0, max(counts_m.values) * 1.2)

# 1c. Stacked — Liver disease vs Mortality
ax = axes[2]
cross = pd.crosstab(liver_slim['liver_disease'], liver_slim['In-hospital_death'])
cross.index = ['No Liver Dis.', 'Liver Dis.']
cross.columns = ['Survived', 'Died']
cross.plot(kind='bar', ax=ax, color=['#4CAF50', '#F44336'],
           edgecolor='white', width=0.5)
ax.set_title('Liver Disease vs Mortality')
ax.set_xlabel('')
ax.set_ylabel('Patient Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# EDA 2: HISTOGRAMS — All Features
# ============================================================

BIOMARKERS = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin',
              'Creatinine', 'BUN', 'Na', 'K', 'HCO3',
              'Platelets', 'Glucose', 'HR', 'NISysABP', 'Age', 'Gender']

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()
fig.suptitle('Feature Distributions — Liver ICU Dataset', fontsize=15, fontweight='bold')

for i, col in enumerate(BIOMARKERS):
    ax = axes[i]
    data = liver_slim[col].dropna()
    ax.hist(data, bins=40, color='#5B9BD5', edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='-',  linewidth=1.5, label=f'Median: {data.median():.1f}')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('histograms.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# EDA 3: BOXPLOTS — Features vs Mortality
# ============================================================

LIVER_COLS = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin',
              'Creatinine', 'BUN', 'Platelets', 'Glucose']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()
fig.suptitle('Boxplots: Key Features vs ICU Mortality', fontsize=15, fontweight='bold')

for i, col in enumerate(LIVER_COLS):
    ax = axes[i]
    data_survived = liver_slim[liver_slim['In-hospital_death'] == 0][col]
    data_died     = liver_slim[liver_slim['In-hospital_death'] == 1][col]

    bp = ax.boxplot([data_survived, data_died],
                    labels=['Survived', 'Died'],
                    patch_artist=True,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='o', markersize=3, alpha=0.4))

    bp['boxes'][0].set_facecolor('#4CAF5066')
    bp['boxes'][1].set_facecolor('#F4433666')

    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Value')

    # Add significance annotation (simple mean difference)
    m1, m2 = data_survived.median(), data_died.median()
    ax.text(0.97, 0.96, f'Δ median: {abs(m2-m1):.2f}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, color='navy',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('boxplots_mortality.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# EDA 4: VIOLIN PLOTS — Liver Biomarkers vs Mortality
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
fig.suptitle('Violin Plots: Liver Biomarkers vs Mortality', fontsize=15, fontweight='bold')

VIOLIN_COLS = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin', 'Creatinine']

for i, col in enumerate(VIOLIN_COLS):
    ax = axes[i]
    plot_df = liver_slim[[col, 'In-hospital_death']].copy()
    plot_df['Outcome'] = plot_df['In-hospital_death'].map({0: 'Survived', 1: 'Died'})

    # Clip extreme outliers for better visualization
    q99 = plot_df[col].quantile(0.99)
    plot_df[col] = plot_df[col].clip(upper=q99)

    sns.violinplot(data=plot_df, x='Outcome', y=col, ax=ax,
                   palette={'Survived': '#4CAF50', 'Died': '#F44336'},
                   inner='box', cut=0)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig('violin_plots.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# EDA 5: CORRELATION HEATMAP
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Correlation Analysis', fontsize=15, fontweight='bold')

feature_cols = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin',
                'Creatinine', 'BUN', 'Na', 'K', 'HCO3',
                'Platelets', 'Glucose', 'HR', 'NISysABP', 'Age',
                'liver_disease', 'In-hospital_death']

corr_matrix = liver_slim[feature_cols].corr()

# Full heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=axes[0],
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
axes[0].set_title('Full Correlation Matrix', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# Correlation with mortality (bar chart)
ax2 = axes[1]
corr_with_target = corr_matrix['In-hospital_death'].drop('In-hospital_death').sort_values()
colors = ['#F44336' if x > 0 else '#4CAF50' for x in corr_with_target.values]
bars = ax2.barh(corr_with_target.index, corr_with_target.values,
                color=colors, edgecolor='white', height=0.6)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title('Correlation with In-hospital Mortality', fontweight='bold')
ax2.set_xlabel('Pearson Correlation Coefficient')
for bar, val in zip(bars, corr_with_target.values):
    ax2.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# EDA 6: LIVER BIOMARKERS vs MORTALITY — Mean Comparison
# ============================================================

liver_biomarkers = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Liver Biomarkers vs Mortality', fontsize=15, fontweight='bold')

# 6a. Mean values by outcome
means = liver_slim.groupby('In-hospital_death')[liver_biomarkers].mean().T
means.columns = ['Survived', 'Died']
means_norm = means.div(means.max(axis=1), axis=0)  # normalize for comparison

x = np.arange(len(liver_biomarkers))
width = 0.35
ax = axes[0]
b1 = ax.bar(x - width/2, means_norm['Survived'], width, label='Survived',
            color='#4CAF50', edgecolor='white')
b2 = ax.bar(x + width/2, means_norm['Died'],     width, label='Died',
            color='#F44336', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(liver_biomarkers, rotation=15)
ax.set_title('Normalized Mean Biomarker Values\nby Outcome')
ax.set_ylabel('Normalized Value (0–1)')
ax.legend()
ax.set_ylim(0, 1.2)

# Add actual values as annotations
for bar, val in zip(b1, means['Survived']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8, color='#2E7D32')
for bar, val in zip(b2, means['Died']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8, color='#B71C1C')

# 6b. Mortality rate by liver disease status + biomarker quartiles
ax2 = axes[1]
liver_slim['Bilirubin_quartile'] = pd.qcut(liver_slim['Bilirubin'], 4,
                                            labels=['Q1\n(Low)', 'Q2', 'Q3', 'Q4\n(High)'])
mort_by_q = liver_slim.groupby('Bilirubin_quartile')['In-hospital_death'].mean() * 100
colors_q = ['#C8E6C9', '#FFF9C4', '#FFCC80', '#EF9A9A']
bars3 = ax2.bar(mort_by_q.index, mort_by_q.values, color=colors_q, edgecolor='white', width=0.5)
for bar, val in zip(bars3, mort_by_q.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
ax2.set_title('Mortality Rate by Bilirubin Quartile\n(Higher Bilirubin = Higher Risk)')
ax2.set_ylabel('Mortality Rate (%)')
ax2.set_xlabel('Bilirubin Level')
ax2.set_ylim(0, max(mort_by_q.values) * 1.25)

plt.tight_layout()
plt.savefig('biomarkers_vs_mortality.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# EDA 7: PAIRPLOT — Core Liver Biomarkers
# ============================================================

# Use subset for pairplot (too many cols = slow)
pair_cols = ['Bilirubin', 'ALT', 'AST', 'Albumin', 'In-hospital_death']
pair_df = liver_slim[pair_cols].copy()
pair_df['Outcome'] = pair_df['In-hospital_death'].map({0: 'Survived', 1: 'Died'})

# Clip extreme outliers for readability
for col in ['Bilirubin', 'ALT', 'AST']:
    pair_df[col] = pair_df[col].clip(upper=pair_df[col].quantile(0.97))

pp = sns.pairplot(pair_df.drop('In-hospital_death', axis=1),
                  hue='Outcome',
                  palette={'Survived': '#4CAF50', 'Died': '#F44336'},
                  plot_kws={'alpha': 0.5, 's': 20},
                  diag_kind='kde',
                  corner=True)
pp.fig.suptitle('Pairplot: Core Liver Biomarkers by Outcome',
                y=1.01, fontsize=14, fontweight='bold')

plt.savefig('pairplot.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# ============================================================
# EDA 8: OUTLIER ANALYSIS — IQR Method
# ============================================================

OUTLIER_COLS = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin',
                'Creatinine', 'BUN', 'Platelets', 'Glucose']

outlier_summary = []
for col in OUTLIER_COLS:
    Q1  = liver_slim[col].quantile(0.25)
    Q3  = liver_slim[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((liver_slim[col] < lower) | (liver_slim[col] > upper)).sum()
    outlier_summary.append({
        'Feature': col, 'Q1': round(Q1, 2), 'Q3': round(Q3, 2),
        'IQR': round(IQR, 2), 'Lower_fence': round(lower, 2),
        'Upper_fence': round(upper, 2),
        'Outliers': n_out,
        'Outlier_%': round(n_out / len(liver_slim) * 100, 1)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier_%', ascending=False)
print("=" * 65)
print("OUTLIER SUMMARY (IQR Method)")
print("=" * 65)
print(outlier_df.to_string(index=False))

# Visual — outlier count bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Outlier Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
colors_out = ['#F44336' if p > 10 else '#FF9800' if p > 5 else '#4CAF50'
              for p in outlier_df['Outlier_%']]
bars = ax.bar(outlier_df['Feature'], outlier_df['Outlier_%'],
              color=colors_out, edgecolor='white', width=0.6)
ax.set_title('Outlier % per Feature (IQR method)')
ax.set_ylabel('Outlier %')
ax.set_xticklabels(outlier_df['Feature'], rotation=30, ha='right')
ax.axhline(5, color='orange', linestyle='--', linewidth=1, label='5% threshold')
ax.axhline(10, color='red', linestyle='--', linewidth=1, label='10% threshold')
ax.legend()
for bar, val in zip(bars, outlier_df['Outlier_%']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val}%', ha='center', va='bottom', fontsize=9)

# Zoomed boxplot to see outliers
ax2 = axes[1]
plot_data = [liver_slim[col].clip(upper=liver_slim[col].quantile(0.99)).dropna()
             for col in OUTLIER_COLS]
bp = ax2.boxplot(plot_data, labels=OUTLIER_COLS, patch_artist=True,
                 flierprops=dict(marker='o', markersize=3, alpha=0.4, color='red'))
for patch in bp['boxes']:
    patch.set_facecolor('#5B9BD566')
ax2.set_title('Boxplot — Outlier Distribution (clipped at 99th pct)')
ax2.set_xticklabels(OUTLIER_COLS, rotation=30, ha='right')
ax2.set_ylabel('Value')

plt.tight_layout()
plt.savefig('outlier_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# EDA SUMMARY
# ============================================================

print("=" * 55)
print("  EDA COMPLETE — SUMMARY")
print("=" * 55)
print(f"Total Patients:        {len(liver_slim)}")
print(f"Features:              {len(BIOMARKERS)}")
print(f"Liver Disease Cases:   {liver_slim['liver_disease'].sum()} ({liver_slim['liver_disease'].mean()*100:.1f}%)")
print(f"ICU Mortality Cases:   {liver_slim['In-hospital_death'].sum()} ({liver_slim['In-hospital_death'].mean()*100:.1f}%)")
print(f"Missing Values:        {liver_slim.isnull().sum().sum()}")
print()
print("Key Findings:")
print("  • Bilirubin, ALT, AST are right-skewed → will need log transform")
print("  • Albumin is negatively correlated with mortality (low = bad)")
print("  • Liver disease patients have significantly higher mortality")
print("  • Several features have >10% outliers (expected in ICU data)")
print()
print("=" * 55)

STEP3:

In [ ]:
# ============================================================
# STEP 2: NORMALITY TESTING & TRANSFORMATIONS
# ============================================================

from scipy import stats
from scipy.stats import shapiro, skew, kurtosis
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_palette("Set2")

# Features to test (exclude IDs and binary targets)
NUMERIC_COLS = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin',
                'Creatinine', 'BUN', 'Na', 'K', 'HCO3',
                'Platelets', 'Glucose', 'HR', 'NISysABP', 'Age']

print("✅ Step 2 imports ready")
print(f"Testing normality for {len(NUMERIC_COLS)} features")

In [ ]:
# ============================================================
# NORMALITY TESTS: Shapiro-Wilk, Skewness, Kurtosis
# ============================================================

normality_results = []

for col in NUMERIC_COLS:
    data = liver_slim[col].dropna()

    # Shapiro-Wilk (use sample of 500 max — SW breaks on large N)
    sample = data.sample(min(500, len(data)), random_state=42)
    stat, p_val = shapiro(sample)

    sk  = skew(data)
    kurt = kurtosis(data)  # excess kurtosis (normal = 0)

    normality_results.append({
        'Feature'        : col,
        'Shapiro_W'      : round(stat, 4),
        'Shapiro_p'      : round(p_val, 6),
        'Normal?'        : '✅ Yes' if p_val > 0.05 else '❌ No',
        'Skewness'       : round(sk, 3),
        'Skew_Type'      : 'Right' if sk > 0.5 else ('Left' if sk < -0.5 else 'Symmetric'),
        'Kurtosis'       : round(kurt, 3),
        'Kurt_Type'      : 'Heavy-tail' if kurt > 1 else ('Light-tail' if kurt < -1 else 'Normal'),
        'Log_Transform?' : '✅ Recommended' if sk > 1.0 else ('⚠️ Optional' if sk > 0.5 else '❌ Not needed')
    })

norm_df = pd.DataFrame(normality_results).sort_values('Skewness', ascending=False)

print("=" * 90)
print("  NORMALITY TEST RESULTS")
print("=" * 90)
print(norm_df.to_string(index=False))
print()
print(f"Features needing log transform (skewness > 1.0):  "
      f"{norm_df[norm_df['Skewness'] > 1.0]['Feature'].tolist()}")
print(f"Features possibly needing transform (0.5–1.0):    "
      f"{norm_df[(norm_df['Skewness'] > 0.5) & (norm_df['Skewness'] <= 1.0)]['Feature'].tolist()}")

In [ ]:
# ============================================================
# QQ PLOTS — Before Transformation
# ============================================================

fig, axes = plt.subplots(3, 5, figsize=(22, 14))
axes = axes.flatten()
fig.suptitle('QQ Plots — Before Transformation\n(Points on diagonal = Normal)',
             fontsize=14, fontweight='bold')

for i, col in enumerate(NUMERIC_COLS):
    ax = axes[i]
    data = liver_slim[col].dropna()
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist="norm")
    ax.plot(osm, osr, 'o', color='#5B9BD5', markersize=3, alpha=0.6)
    ax.plot(osm, slope * np.array(osm) + intercept,
            'r-', linewidth=1.5, label=f'R²={r**2:.3f}')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('Theoretical Quantiles')
    ax.set_ylabel('Sample Quantiles')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('qq_plots_before.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ============================================================
# LOG TRANSFORMATION
# Columns with skewness > 0.5 get log1p transform
# ============================================================

liver_transformed = liver_slim.copy()

# Auto-select columns needing log transform based on skewness > 0.5
LOG_COLS = norm_df[norm_df['Skewness'] > 0.5]['Feature'].tolist()
print(f"Applying log1p transform to: {LOG_COLS}\n")

for col in LOG_COLS:
    # Shift if any non-positive values exist
    min_val = liver_transformed[col].min()
    if min_val <= 0:
        shift = abs(min_val) + 1e-6
        liver_transformed[f'{col}_log'] = np.log1p(liver_transformed[col] + shift)
        print(f"  {col}: shifted by {shift:.4f} then log1p applied")
    else:
        liver_transformed[f'{col}_log'] = np.log1p(liver_transformed[col])
        print(f"  {col}: log1p applied directly")

LOG_COLS_NEW = [f'{c}_log' for c in LOG_COLS]

# Verify skewness improvement
print("\n--- Skewness Before vs After Log Transform ---")
print(f"{'Feature':<15} {'Before':>10} {'After':>10} {'Improved?':>12}")
print("-" * 50)
for col, col_log in zip(LOG_COLS, LOG_COLS_NEW):
    before = round(skew(liver_slim[col].dropna()), 3)
    after  = round(skew(liver_transformed[col_log].dropna()), 3)
    improved = '✅ Yes' if abs(after) < abs(before) else '❌ No'
    print(f"{col:<15} {before:>10} {after:>10} {improved:>12}")

print(f"\n Log-transformed columns created: {LOG_COLS_NEW}")

In [ ]:
# ============================================================
# BEFORE vs AFTER TRANSFORMATION — Visual Comparison
# ============================================================

SHOW_COLS = LOG_COLS[:6]  # Show top 6 transformed features

fig, axes = plt.subplots(len(SHOW_COLS), 2, figsize=(14, len(SHOW_COLS) * 3))
fig.suptitle('Distribution: Before vs After Log Transform', fontsize=14, fontweight='bold')

for i, col in enumerate(SHOW_COLS):
    col_log = f'{col}_log'

    # Before
    ax1 = axes[i][0]
    ax1.hist(liver_slim[col].dropna(), bins=40,
             color='#F44336', edgecolor='white', alpha=0.8)
    ax1.set_title(f'{col} — BEFORE (skew={skew(liver_slim[col].dropna()):.2f})',
                  fontweight='bold', fontsize=10)
    ax1.set_ylabel('Count')
    ax1.axvline(liver_slim[col].mean(),   color='black', linestyle='--', lw=1.5)
    ax1.axvline(liver_slim[col].median(), color='blue',  linestyle='-',  lw=1.5)

    # After
    ax2 = axes[i][1]
    ax2.hist(liver_transformed[col_log].dropna(), bins=40,
             color='#4CAF50', edgecolor='white', alpha=0.8)
    sk_after = skew(liver_transformed[col_log].dropna())
    ax2.set_title(f'{col}_log — AFTER (skew={sk_after:.2f})',
                  fontweight='bold', fontsize=10)
    ax2.set_ylabel('Count')
    ax2.axvline(liver_transformed[col_log].mean(),   color='black', linestyle='--', lw=1.5)
    ax2.axvline(liver_transformed[col_log].median(), color='blue',  linestyle='-',  lw=1.5)

plt.tight_layout()
plt.savefig('transform_comparison.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ============================================================
# SCALING COMPARISON: StandardScaler vs MinMaxScaler
# Applied to log-transformed + remaining numeric features
# ============================================================

# Build clean feature set for scaling
# Use log versions for skewed cols, originals for symmetric cols
SYMMETRIC_COLS = [c for c in NUMERIC_COLS if c not in LOG_COLS]
FEATURE_SET    = SYMMETRIC_COLS + LOG_COLS_NEW

print(f"Symmetric cols (no transform): {SYMMETRIC_COLS}")
print(f"Log-transformed cols:          {LOG_COLS_NEW}")
print(f"Total feature set size:        {len(FEATURE_SET)}")

scale_data = liver_transformed[FEATURE_SET].dropna()

# Apply both scalers
ss  = StandardScaler()
mms = MinMaxScaler()

data_std    = pd.DataFrame(ss.fit_transform(scale_data),
                           columns=FEATURE_SET)
data_minmax = pd.DataFrame(mms.fit_transform(scale_data),
                           columns=FEATURE_SET)

# Comparison table
comparison = pd.DataFrame({
    'Feature'  : FEATURE_SET,
    'Original_Mean'  : scale_data.mean().round(3).values,
    'Original_Std'   : scale_data.std().round(3).values,
    'Standard_Mean'  : data_std.mean().round(3).values,
    'Standard_Std'   : data_std.std().round(3).values,
    'MinMax_Min'     : data_minmax.min().round(3).values,
    'MinMax_Max'     : data_minmax.max().round(3).values,
})

print("\n" + "=" * 80)
print("  SCALING COMPARISON TABLE")
print("=" * 80)
print(comparison.to_string(index=False))

In [ ]:
# ============================================================
# SCALING VISUAL — Compare Original, StandardScaler, MinMaxScaler
# ============================================================

SHOW = FEATURE_SET[:6]  # show first 6 features

fig, axes = plt.subplots(3, 6, figsize=(22, 10))
fig.suptitle('Scaling Comparison: Original vs StandardScaler vs MinMaxScaler',
             fontsize=13, fontweight='bold')

row_labels = ['Original', 'StandardScaler', 'MinMaxScaler']
datasets   = [scale_data, data_std, data_minmax]
row_colors = ['#5B9BD5', '#FF9800', '#4CAF50']

for row, (label, df, color) in enumerate(zip(row_labels, datasets, row_colors)):
    for col_i, col in enumerate(SHOW):
        ax = axes[row][col_i]
        ax.hist(df[col].dropna(), bins=30, color=color, edgecolor='white', alpha=0.85)
        if col_i == 0:
            ax.set_ylabel(label, fontweight='bold', fontsize=10)
        if row == 0:
            # Clean column name for title
            title = col.replace('_log', '\n(log)')
            ax.set_title(title, fontweight='bold', fontsize=9)
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('scaling_comparison.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ============================================================
# SAVE FINAL TRANSFORMED DATAFRAME FOR NEXT STEPS
# ============================================================

# Build final modeling dataframe:
# - Use log-transformed versions for skewed features
# - Keep original versions for symmetric features
# - Keep targets: liver_disease, In-hospital_death
# - Drop RecordID (not a feature)

# Replace skewed originals with their log versions in one dataframe
liver_model = liver_slim.copy()

for col in LOG_COLS:
    liver_model[col] = liver_transformed[f'{col}_log'].values

# Drop RecordID
liver_model = liver_model.drop(columns=['RecordID'], errors='ignore')

print("=" * 55)
print("  FINAL TRANSFORMED DATASET READY")
print("=" * 55)
print(f"Shape:          {liver_model.shape}")
print(f"Missing values: {liver_model.isnull().sum().sum()}")
print(f"\nColumns: {list(liver_model.columns)}")
print()
print(f"Targets:")
print(f"  liver_disease:     {liver_model['liver_disease'].value_counts().to_dict()}")
print(f"  In-hospital_death: {liver_model['In-hospital_death'].value_counts().to_dict()}")
print()
print("Log-transformed features (stored in-place):")
for col in LOG_COLS:
    sk_after = round(skew(liver_model[col].dropna()), 3)
    print(f"  {col:<15} → skewness now: {sk_after}")

print()
print("=" * 55)

STEP 3 Feature Selection

In [ ]:
# ============================================================
# STEP 3: FEATURE SELECTION
# ============================================================

from sklearn.feature_selection import (
    chi2, mutual_info_classif, SelectKBest, f_classif
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBClassifier
from scipy.stats import spearmanr
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_palette("Set2")

# ── Feature columns (log-transformed versions already in liver_model) ──
FEATURE_COLS = [
    'Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin',
    'Creatinine', 'BUN', 'Na', 'K', 'HCO3',
    'Platelets', 'Glucose', 'HR', 'NISysABP',
    'Age', 'Gender'
]

TARGET_LIVER    = 'liver_disease'
TARGET_MORT     = 'In-hospital_death'

X = liver_model[FEATURE_COLS].copy()
y_liver = liver_model[TARGET_LIVER].copy()
y_mort  = liver_model[TARGET_MORT].copy()

print("✅ Feature Selection setup ready")
print(f"Feature matrix shape : {X.shape}")
print(f"Liver disease target : {y_liver.value_counts().to_dict()}")
print(f"Mortality target     : {y_mort.value_counts().to_dict()}")

In [ ]:
# ============================================================
# 3.1 CORRELATION ANALYSIS — Spearman (robust to non-normality)
# ============================================================

# Spearman correlation with both targets
spearman_results = []

for col in FEATURE_COLS:
    r_liver, p_liver = spearmanr(X[col], y_liver)
    r_mort,  p_mort  = spearmanr(X[col], y_mort)
    spearman_results.append({
        'Feature'         : col,
        'Spearman_Liver'  : round(r_liver, 4),
        'p_Liver'         : round(p_liver, 5),
        'Sig_Liver'       : '✅' if p_liver < 0.05 else '❌',
        'Spearman_Mort'   : round(r_mort, 4),
        'p_Mort'          : round(p_mort, 5),
        'Sig_Mort'        : '✅' if p_mort < 0.05 else '❌',
    })

spearman_df = pd.DataFrame(spearman_results).sort_values(
    'Spearman_Mort', key=abs, ascending=False
)

print("=" * 80)
print("  SPEARMAN CORRELATION — Features vs Targets")
print("=" * 80)
print(spearman_df.to_string(index=False))

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Spearman Correlation with Targets', fontsize=14, fontweight='bold')

for ax, col_r, col_p, title, color_pos, color_neg in [
    (axes[0], 'Spearman_Liver', 'p_Liver',
     'Correlation with Liver Disease', '#1976D2', '#E53935'),
    (axes[1], 'Spearman_Mort',  'p_Mort',
     'Correlation with Mortality',     '#E53935', '#1976D2'),
]:
    df_sorted = spearman_df.sort_values(col_r)
    colors = [color_pos if v > 0 else color_neg for v in df_sorted[col_r]]
    bars = ax.barh(df_sorted['Feature'], df_sorted[col_r],
                   color=colors, edgecolor='white', height=0.6)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Spearman ρ')
    for bar, val, p in zip(bars, df_sorted[col_r], df_sorted[col_p]):
        marker = '*' if p < 0.05 else ''
        ax.text(val + (0.005 if val >= 0 else -0.005),
                bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}{marker}', va='center',
                ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig('spearman_correlation.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ============================================================
# 3.2 MULTICOLLINEARITY — Variance Inflation Factor (VIF)
# ============================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X_vif = add_constant(X.astype(float))

vif_data = pd.DataFrame()
vif_data['Feature'] = X.columns
vif_data['VIF']     = [
    variance_inflation_factor(X_vif.values, i + 1)   # +1 skips constant
    for i in range(len(X.columns))
]
vif_data['VIF']        = vif_data['VIF'].round(3)
vif_data['Risk_Level'] = vif_data['VIF'].apply(
    lambda v: '🔴 High (>10)'   if v > 10
    else ('🟠 Moderate (5–10)' if v > 5
    else '🟢 Low (<5)')
)
vif_data = vif_data.sort_values('VIF', ascending=False)

print("=" * 55)
print("  VARIANCE INFLATION FACTOR (VIF)")
print("  VIF > 10 → remove; 5–10 → monitor; <5 → safe")
print("=" * 55)
print(vif_data.to_string(index=False))

# ── Plot ──
fig, ax = plt.subplots(figsize=(10, 6))
colors_vif = [
    '#F44336' if v > 10 else '#FF9800' if v > 5 else '#4CAF50'
    for v in vif_data['VIF']
]
bars = ax.barh(vif_data['Feature'], vif_data['VIF'],
               color=colors_vif, edgecolor='white', height=0.6)
ax.axvline(5,  color='orange', linestyle='--', linewidth=1.5, label='VIF=5 (warn)')
ax.axvline(10, color='red',    linestyle='--', linewidth=1.5, label='VIF=10 (high)')
ax.set_title('Variance Inflation Factor — Multicollinearity Check',
             fontweight='bold', fontsize=13)
ax.set_xlabel('VIF Score')
ax.legend()
for bar, val in zip(bars, vif_data['VIF']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('vif_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

HIGH_VIF = vif_data[vif_data['VIF'] > 10]['Feature'].tolist()
print(f"\n🔴 Features with HIGH VIF (>10): {HIGH_VIF}")
print("These will be monitored but kept — clinically meaningful in ICU context")

In [ ]:
# ============================================================
# 3.3 CHI-SQUARE FEATURE SELECTION
# Requires non-negative values → use MinMaxScaler first
# ============================================================

mms = MinMaxScaler()
X_scaled = pd.DataFrame(mms.fit_transform(X), columns=FEATURE_COLS)

# ── Chi-Square vs Liver Disease ──
chi2_liver_scores, chi2_liver_pvals = chi2(X_scaled, y_liver)
# ── Chi-Square vs Mortality ──
chi2_mort_scores,  chi2_mort_pvals  = chi2(X_scaled, y_mort)

chi2_df = pd.DataFrame({
    'Feature'        : FEATURE_COLS,
    'Chi2_Liver'     : chi2_liver_scores.round(4),
    'p_Liver'        : chi2_liver_pvals.round(6),
    'Sig_Liver'      : ['✅' if p < 0.05 else '❌' for p in chi2_liver_pvals],
    'Chi2_Mort'      : chi2_mort_scores.round(4),
    'p_Mort'         : chi2_mort_pvals.round(6),
    'Sig_Mort'       : ['✅' if p < 0.05 else '❌' for p in chi2_mort_pvals],
}).sort_values('Chi2_Mort', ascending=False)

print("=" * 80)
print("  CHI-SQUARE FEATURE SCORES")
print("=" * 80)
print(chi2_df.to_string(index=False))

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Chi-Square Scores — Feature Importance', fontsize=14, fontweight='bold')

for ax, score_col, title, color in [
    (axes[0], 'Chi2_Liver', 'vs Liver Disease', '#1976D2'),
    (axes[1], 'Chi2_Mort',  'vs Mortality',     '#E53935'),
]:
    df_s = chi2_df.sort_values(score_col)
    bars = ax.barh(df_s['Feature'], df_s[score_col],
                   color=color, alpha=0.8, edgecolor='white', height=0.6)
    ax.set_title(f'Chi-Square {title}', fontweight='bold')
    ax.set_xlabel('Chi² Score')
    for bar, val in zip(bars, df_s[score_col]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('chi2_scores.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ============================================================
# 3.4 MUTUAL INFORMATION
# ============================================================

mi_liver = mutual_info_classif(X_scaled, y_liver, random_state=42)
mi_mort  = mutual_info_classif(X_scaled, y_mort,  random_state=42)

mi_df = pd.DataFrame({
    'Feature'     : FEATURE_COLS,
    'MI_Liver'    : mi_liver.round(5),
    'MI_Mortality': mi_mort.round(5),
}).sort_values('MI_Mortality', ascending=False)

print("=" * 50)
print("  MUTUAL INFORMATION SCORES")
print("=" * 50)
print(mi_df.to_string(index=False))

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Mutual Information — Feature Relevance', fontsize=14, fontweight='bold')

for ax, col, title, color in [
    (axes[0], 'MI_Liver',     'vs Liver Disease', '#1976D2'),
    (axes[1], 'MI_Mortality', 'vs Mortality',     '#E53935'),
]:
    df_s = mi_df.sort_values(col)
    bars = ax.barh(df_s['Feature'], df_s[col],
                   color=color, alpha=0.8, edgecolor='white', height=0.6)
    ax.set_title(f'Mutual Information {title}', fontweight='bold')
    ax.set_xlabel('MI Score')
    for bar, val in zip(bars, df_s[col]):
        ax.text(bar.get_width() + 0.0005,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('mutual_information.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 3.5 RANDOM FOREST IMPORTANCE
# ============================================================

from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
X_std = pd.DataFrame(ss.fit_transform(X), columns=FEATURE_COLS)

# ── RF for Liver Disease ──
rf_liver = RandomForestClassifier(
    n_estimators=300, max_depth=8,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_liver.fit(X_std, y_liver)

# ── RF for Mortality ──
rf_mort = RandomForestClassifier(
    n_estimators=300, max_depth=8,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_mort.fit(X_std, y_mort)

rf_df = pd.DataFrame({
    'Feature'          : FEATURE_COLS,
    'RF_Liver'         : rf_liver.feature_importances_.round(5),
    'RF_Mortality'     : rf_mort.feature_importances_.round(5),
}).sort_values('RF_Mortality', ascending=False)

print("=" * 50)
print("  RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 50)
print(rf_df.to_string(index=False))

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Random Forest Feature Importance', fontsize=14, fontweight='bold')

for ax, col, title, color in [
    (axes[0], 'RF_Liver',     'Liver Disease', '#1976D2'),
    (axes[1], 'RF_Mortality', 'Mortality',     '#E53935'),
]:
    df_s = rf_df.sort_values(col)
    bars = ax.barh(df_s['Feature'], df_s[col],
                   color=color, alpha=0.85, edgecolor='white', height=0.6)
    ax.set_title(f'RF Importance — {title}', fontweight='bold')
    ax.set_xlabel('Importance Score')
    for bar, val in zip(bars, df_s[col]):
        ax.text(bar.get_width() + 0.0005,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('rf_importance.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 3.6 XGBOOST FEATURE IMPORTANCE
# ============================================================

# ── XGB for Liver Disease ──
xgb_liver = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=(y_liver == 0).sum() / (y_liver == 1).sum(),
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
xgb_liver.fit(X_std, y_liver)

# ── XGB for Mortality ──
xgb_mort = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=(y_mort == 0).sum() / (y_mort == 1).sum(),
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
xgb_mort.fit(X_std, y_mort)

xgb_df = pd.DataFrame({
    'Feature'      : FEATURE_COLS,
    'XGB_Liver'    : xgb_liver.feature_importances_.round(5),
    'XGB_Mortality': xgb_mort.feature_importances_.round(5),
}).sort_values('XGB_Mortality', ascending=False)

print("=" * 50)
print("  XGBOOST FEATURE IMPORTANCE")
print("=" * 50)
print(xgb_df.to_string(index=False))

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('XGBoost Feature Importance', fontsize=14, fontweight='bold')

for ax, col, title, color in [
    (axes[0], 'XGB_Liver',     'Liver Disease', '#1976D2'),
    (axes[1], 'XGB_Mortality', 'Mortality',     '#E53935'),
]:
    df_s = xgb_df.sort_values(col)
    bars = ax.barh(df_s['Feature'], df_s[col],
                   color=color, alpha=0.85, edgecolor='white', height=0.6)
    ax.set_title(f'XGB Importance — {title}', fontweight='bold')
    ax.set_xlabel('Importance Score')
    for bar, val in zip(bars, df_s[col]):
        ax.text(bar.get_width() + 0.0005,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('xgb_importance.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 3.7 UNIFIED FEATURE RANKING — All Methods Combined
# ============================================================

from sklearn.preprocessing import MinMaxScaler as MMS

def rank_normalize(series):
    """Rank then normalize to 0-1"""
    ranked = series.rank(ascending=True)
    return (ranked - ranked.min()) / (ranked.max() - ranked.min())

# Build unified ranking for MORTALITY (primary target)
ranking = pd.DataFrame({'Feature': FEATURE_COLS})

ranking['Spearman'] = ranking['Feature'].map(
    dict(zip(spearman_df['Feature'], spearman_df['Spearman_Mort'].abs()))
)
ranking['Chi2']  = ranking['Feature'].map(
    dict(zip(chi2_df['Feature'], chi2_df['Chi2_Mort']))
)
ranking['MI']    = ranking['Feature'].map(
    dict(zip(mi_df['Feature'], mi_df['MI_Mortality']))
)
ranking['RF']    = ranking['Feature'].map(
    dict(zip(rf_df['Feature'], rf_df['RF_Mortality']))
)
ranking['XGB']   = ranking['Feature'].map(
    dict(zip(xgb_df['Feature'], xgb_df['XGB_Mortality']))
)

# Normalize each method score to 0-1 for fair combination
for col in ['Spearman', 'Chi2', 'MI', 'RF', 'XGB']:
    ranking[f'{col}_norm'] = rank_normalize(ranking[col])

# Composite score = equal-weight average of all 5 methods
norm_cols = ['Spearman_norm', 'Chi2_norm', 'MI_norm', 'RF_norm', 'XGB_norm']
ranking['Composite_Score'] = ranking[norm_cols].mean(axis=1).round(4)
ranking = ranking.sort_values('Composite_Score', ascending=False).reset_index(drop=True)
ranking.index += 1  # rank starts at 1

print("=" * 75)
print("  FINAL FEATURE RANKING — MORTALITY PREDICTION (All Methods Combined)")
print("=" * 75)
display_cols = ['Feature', 'Spearman', 'Chi2', 'MI', 'RF', 'XGB', 'Composite_Score']
print(ranking[display_cols].to_string())

# ── Select top features ──
TOP_N = 12
SELECTED_FEATURES = ranking['Feature'].head(TOP_N).tolist()
print(f"\n{'=' * 55}")
print(f"  SELECTED TOP {TOP_N} FEATURES FOR MODELING")
print(f"{'=' * 55}")
for i, f in enumerate(SELECTED_FEATURES, 1):
    score = ranking[ranking['Feature'] == f]['Composite_Score'].values[0]
    print(f"  {i:>2}. {f:<18}  composite score: {score:.4f}")

# ── Heatmap of normalized scores ──
fig, ax = plt.subplots(figsize=(12, 7))
heat_data = ranking[['Feature'] + norm_cols + ['Composite_Score']].set_index('Feature')
heat_data.columns = ['Spearman', 'Chi²', 'MI', 'RF', 'XGB', 'Composite']

sns.heatmap(heat_data, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Normalized Score'},
            annot_kws={'size': 9})
ax.set_title('Feature Importance Heatmap — All Methods\n(Mortality Prediction)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('feature_ranking_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# SAVE SELECTED FEATURES FOR NEXT STEPS
# ============================================================

# Final feature sets for both tasks
LIVER_FEATURES = SELECTED_FEATURES.copy()   # same set for liver disease model
MORT_FEATURES  = SELECTED_FEATURES.copy()   # same set for mortality model

# Final X and y matrices — ready for train/test split
X_liver_final = liver_model[LIVER_FEATURES]
y_liver_final = liver_model['liver_disease']

X_mort_final  = liver_model[MORT_FEATURES]
y_mort_final  = liver_model['In-hospital_death']

print("=" * 55)
print("  STEP 3 COMPLETE — FEATURE SELECTION DONE")
print("=" * 55)
print(f"\nSelected Features ({TOP_N}): {SELECTED_FEATURES}")
print(f"\nX_liver_final shape : {X_liver_final.shape}")
print(f"y_liver_final dist  : {y_liver_final.value_counts().to_dict()}")
print(f"\nX_mort_final shape  : {X_mort_final.shape}")
print(f"y_mort_final dist   : {y_mort_final.value_counts().to_dict()}")
print()
print("=" * 55)

STEP 4: Class imbalance handling

In [ ]:
# ============================================================
# STEP 4: CLASS IMBALANCE HANDLING
# ============================================================

from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# ── Manually add Albumin back (clinically critical liver biomarker) ──
SELECTED_FEATURES = [
    'BUN', 'HCO3', 'Age', 'Bilirubin', 'Glucose', 'Creatinine',
    'ALP', 'HR', 'ALT', 'Platelets', 'NISysABP', 'AST', 'Albumin'
]

X_liver_final = liver_model[SELECTED_FEATURES].copy()
y_liver_final = liver_model['liver_disease'].copy()

X_mort_final  = liver_model[SELECTED_FEATURES].copy()
y_mort_final  = liver_model['In-hospital_death'].copy()

print("=" * 55)
print("  FEATURE SET FINALIZED (13 features)")
print("=" * 55)
print(f"Features: {SELECTED_FEATURES}")
print()

# ── Imbalance check ──
def imbalance_report(y, name):
    counts = Counter(y)
    total  = len(y)
    minority = min(counts.values())
    majority = max(counts.values())
    ratio    = majority / minority
    print(f"  Target          : {name}")
    print(f"  Class 0         : {counts[0]}  ({counts[0]/total*100:.1f}%)")
    print(f"  Class 1         : {counts[1]}  ({counts[1]/total*100:.1f}%)")
    print(f"  Imbalance Ratio : {ratio:.2f}:1")
    if ratio > 3:
        print(f"  ⚠️  Imbalanced — SMOTE recommended")
    else:
        print(f"  ✅  Balanced enough — SMOTE optional")
    print()
    return ratio

print("─" * 55)
ratio_liver = imbalance_report(y_liver_final, 'Liver Disease')
ratio_mort  = imbalance_report(y_mort_final,  'In-hospital Mortality')
print("─" * 55)

In [ ]:
# ============================================================
# VISUALIZE CLASS IMBALANCE — Before SMOTE
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Class Imbalance Analysis — Before SMOTE', fontsize=14, fontweight='bold')

# ── Plot 1: Liver Disease ──
ax = axes[0]
counts_l = y_liver_final.value_counts().sort_index()
labels_l = ['No Liver Disease (0)', 'Liver Disease (1)']
colors_l = ['#4CAF50', '#F44336']
bars = ax.bar(labels_l, counts_l.values, color=colors_l, width=0.5, edgecolor='white')
for bar, val in zip(bars, counts_l.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val}\n({val/len(y_liver_final)*100:.1f}%)',
            ha='center', va='bottom', fontweight='bold')
ax.set_title('Liver Disease', fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, max(counts_l.values) * 1.25)
ax.set_xticklabels(labels_l, rotation=10)

# ── Plot 2: Mortality ──
ax = axes[1]
counts_m = y_mort_final.value_counts().sort_index()
labels_m = ['Survived (0)', 'Died (1)']
colors_m = ['#2196F3', '#FF5722']
bars2 = ax.bar(labels_m, counts_m.values, color=colors_m, width=0.5, edgecolor='white')
for bar, val in zip(bars2, counts_m.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val}\n({val/len(y_mort_final)*100:.1f}%)',
            ha='center', va='bottom', fontweight='bold')
ax.set_title('ICU Mortality', fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, max(counts_m.values) * 1.25)

# ── Plot 3: Pie comparison ──
ax = axes[2]
ratios = {
    'Liver Disease\n(64.6% vs 35.4%)': [1131, 619],
    'Mortality\n(80.9% vs 19.1%)':     [1415, 335],
}
x_pos  = [0.25, 0.75]
colors_pie = [['#F44336', '#4CAF50'], ['#FF5722', '#2196F3']]
for pos, (label, vals), cols in zip(x_pos, ratios.items(), colors_pie):
    wedges, texts, autotexts = ax.pie(
        vals, labels=None, colors=cols,
        autopct='%1.1f%%', startangle=90,
        radius=0.35, center=(pos, 0.5),
        wedgeprops=dict(edgecolor='white', linewidth=1.5),
        textprops=dict(fontsize=9)
    )
    ax.text(pos, 0.08, label, ha='center', va='center',
            fontsize=8, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Class Ratio Comparison', fontweight='bold')

plt.tight_layout()
plt.savefig('imbalance_before.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# STEP 4 CELL 3 — CORRECT SMOTE
# Split FIRST, then scale, then SMOTE only on train
# ============================================================

from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from collections import Counter
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Step 1: Original data ──
X_orig      = liver_model[SELECTED_FEATURES].copy()
y_orig_liv  = liver_model['liver_disease'].copy()
y_orig_mort = liver_model['In-hospital_death'].copy()

# ── Step 2: Split BEFORE scaling or SMOTE ──
X_tr_liv,  X_te_liv,  y_tr_liv,  y_te_liv  = train_test_split(
    X_orig, y_orig_liv,
    test_size=0.2, stratify=y_orig_liv, random_state=42
)
X_tr_mort, X_te_mort, y_tr_mort, y_te_mort = train_test_split(
    X_orig, y_orig_mort,
    test_size=0.2, stratify=y_orig_mort, random_state=42
)

# ── Step 3: Fit scaler on train only, transform both ──
scaler_liv  = StandardScaler()
scaler_mort = StandardScaler()

X_tr_liv_sc  = pd.DataFrame(scaler_liv.fit_transform(X_tr_liv),
                             columns=SELECTED_FEATURES)
X_te_liv_sc  = pd.DataFrame(scaler_liv.transform(X_te_liv),
                             columns=SELECTED_FEATURES)

X_tr_mort_sc = pd.DataFrame(scaler_mort.fit_transform(X_tr_mort),
                              columns=SELECTED_FEATURES)
X_te_mort_sc = pd.DataFrame(scaler_mort.transform(X_te_mort),
                              columns=SELECTED_FEATURES)

# ── Step 4: SMOTE on training data ONLY ──
smote_liv  = SMOTE(k_neighbors=5, random_state=42)
smote_mort = BorderlineSMOTE(k_neighbors=5, random_state=42, kind='borderline-1')

X_tr_liv_sm,  y_tr_liv_sm  = smote_liv.fit_resample(X_tr_liv_sc,  y_tr_liv)
X_tr_mort_sm, y_tr_mort_sm = smote_mort.fit_resample(X_tr_mort_sc, y_tr_mort)

X_tr_liv_sm  = pd.DataFrame(X_tr_liv_sm,  columns=SELECTED_FEATURES)
X_tr_mort_sm = pd.DataFrame(X_tr_mort_sm, columns=SELECTED_FEATURES)
y_tr_liv_sm  = pd.Series(y_tr_liv_sm,  name='liver_disease')
y_tr_mort_sm = pd.Series(y_tr_mort_sm, name='In-hospital_death')

# ── Summary ──
print("=" * 62)
print("  CORRECT PIPELINE: Split → Scale → SMOTE(train only)")
print("=" * 62)
print(f"\n  LIVER DISEASE:")
print(f"    Train (SMOTE) : {X_tr_liv_sm.shape}  | {Counter(y_tr_liv_sm)}")
print(f"    Test (original): {X_te_liv_sc.shape}  | {Counter(y_te_liv)}")
print(f"    Synthetic added: {len(y_tr_liv_sm) - len(y_tr_liv)}")

print(f"\n  MORTALITY:")
print(f"    Train (SMOTE) : {X_tr_mort_sm.shape}  | {Counter(y_tr_mort_sm)}")
print(f"    Test (original): {X_te_mort_sc.shape}  | {Counter(y_te_mort)}")
print(f"    Mortality rate in test: {y_te_mort.mean()*100:.1f}%  ← real-world")
print(f"    Synthetic added: {len(y_tr_mort_sm) - len(y_tr_mort)}")
print("=" * 62)

In [ ]:
# ============================================================
# STEP 4 COMPLETE — SAVE ALL VARIABLES FOR NEXT STEPS
# ============================================================

print("=" * 60)
print("  STEP 4 COMPLETE — CLASS IMBALANCE ")
print("=" * 60)

print(f"""
  ┌─────────────────────────────────────────────────────┐
  │              LIVER DISEASE TARGET                   │
  │  Before SMOTE : Class0={Counter(y_liver_final)[0]}, Class1={Counter(y_liver_final)[1]}          │
  │  After  SMOTE : Class0={Counter(y_liver_sm)[0]}, Class1={Counter(y_liver_sm)[1]}          │
  │  Method       : SMOTE (regular)                     │
  ├─────────────────────────────────────────────────────┤
  │              MORTALITY TARGET                       │
  │  Before SMOTE : Class0={Counter(y_mort_final)[0]}, Class1={Counter(y_mort_final)[1]}           │
  │  After  SMOTE : Class0={Counter(y_mort_sm)[0]}, Class1={Counter(y_mort_sm)[1]}          │
  │  Method       : BorderlineSMOTE                     │
  └─────────────────────────────────────────────────────┘
""")

print(f"  Final Features ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}")
print()
print(f"  Ready datasets:")
print(f"    X_liver_sm : {X_liver_sm.shape}  | y_liver_sm : {Counter(y_liver_sm)}")
print(f"    X_mort_sm  : {X_mort_sm.shape}   | y_mort_sm  : {Counter(y_mort_sm)}")
print()
print("=" * 60)

In [ ]:
# ============================================================
# STEP 5: SPLIT SUMMARY
# ============================================================

print("=" * 62)
print("  STEP 5: STRATIFIED SPLIT — CONFIRMATION")
print("=" * 62)

for name, X_tr, X_te, y_tr, y_te in [
    ("LIVER DISEASE", X_tr_liv_sm, X_te_liv_sc, y_tr_liv_sm, y_te_liv),
    ("MORTALITY",     X_tr_mort_sm, X_te_mort_sc, y_tr_mort_sm, y_te_mort),
]:
    print(f"\n  {name}:")
    print(f"    Train (SMOTE) shape : {X_tr.shape}")
    print(f"    Test  (original) shape: {X_te.shape}")
    train_rate = y_tr.mean() * 100
    test_rate  = y_te.mean() * 100
    print(f"    Train positive rate : {train_rate:.1f}%")
    print(f"    Test  positive rate : {test_rate:.1f}%  ← real-world distribution")

print("=" * 62)

Step 5 — Train Test Split + Step 6 — ML Models

In [ ]:
# ============================================================
# STEP 6 CELL 1 — DEFINE ALL MODELS
# No scaler in model (already scaled externally)
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve
)
import matplotlib.gridspec as gridspec

def get_models():
    return {
        'Logistic Regression': LogisticRegression(
            max_iter=1000, class_weight='balanced',
            solver='lbfgs', random_state=42
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=300, max_depth=8,
            class_weight='balanced', random_state=42, n_jobs=-1
        ),
        'XGBoost': XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            use_label_encoder=False, eval_metric='logloss',
            random_state=42, n_jobs=-1
        ),
        'SVM': SVC(
            kernel='rbf', C=1.0, gamma='scale',
            class_weight='balanced', probability=True,
            random_state=42
        ),
        'KNN': KNeighborsClassifier(
            n_neighbors=7, weights='distance', n_jobs=-1
        ),
        'Naive Bayes': GaussianNB(),
    }

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("✅ All model definitions ready")
print("   Models: Logistic Regression, Random Forest, XGBoost,")
print("           SVM, KNN, Naive Bayes")

In [ ]:
# ============================================================
# COMPLETE CLEAN FIX — Run this single cell
# Redefines function + trains both tasks fresh
# ============================================================

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    classification_report
)
import pandas as pd
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def train_evaluate_clean(models,
                          X_train, y_train,
                          X_test,  y_test,
                          task_name):
    """
    X_train, y_train : SMOTE balanced (for fitting)
    X_test,  y_test  : Original imbalanced (for evaluation)
    """
    # Safety: force all targets to 1D numpy arrays
    y_train = np.array(y_train).ravel()
    y_test  = np.array(y_test).ravel()

    # Safety: force X to numpy
    X_train = np.array(X_train)
    X_test  = np.array(X_test)

    print(f"\n{'='*65}")
    print(f"  TRAINING: {task_name}")
    print(f"  Train shape: {X_train.shape} | y_train unique: {np.unique(y_train)}")
    print(f"  Test  shape: {X_test.shape}  | y_test  unique: {np.unique(y_test)}")
    print(f"{'='*65}")

    results = []
    trained = {}

    for i, (name, model) in enumerate(models.items(), 1):
        print(f"  [{i}/{len(models)}] {name}...", end=' ')

        # Train
        model.fit(X_train, y_train)

        # Predict on original test
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        acc    = accuracy_score(y_test, y_pred)
        prec   = precision_score(y_test, y_pred, zero_division=0)
        rec    = recall_score(y_test, y_pred, zero_division=0)
        f1     = f1_score(y_test, y_pred, zero_division=0)
        roc    = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)

        # CV on SMOTE train
        cv_sc = cross_val_score(
            model, X_train, y_train,
            cv=cv, scoring='roc_auc', n_jobs=-1
        )

        results.append({
            'Model'       : name,
            'Accuracy'    : round(acc,    4),
            'Precision'   : round(prec,   4),
            'Recall'      : round(rec,    4),
            'F1-Score'    : round(f1,     4),
            'ROC-AUC'     : round(roc,    4),
            'PR-AUC'      : round(pr_auc, 4),
            'CV-AUC Mean' : round(cv_sc.mean(), 4),
            'CV-AUC Std'  : round(cv_sc.std(),  4),
        })
        trained[name] = model
        print(f"ROC-AUC={roc:.4f} | CV={cv_sc.mean():.4f}±{cv_sc.std():.4f}")

    df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
    df.index += 1
    return df, trained

print("✅ Clean function defined — running both tasks now...")
print()

# ── Convert everything to numpy cleanly ──
Xtr_liv  = np.array(X_tr_liv_sm);   ytr_liv  = np.array(y_tr_liv_sm).ravel()
Xte_liv  = np.array(X_te_liv_sc);   yte_liv  = np.array(y_te_liv).ravel()

Xtr_mort = np.array(X_tr_mort_sm);  ytr_mort = np.array(y_tr_mort_sm).ravel()
Xte_mort = np.array(X_te_mort_sc);  yte_mort = np.array(y_te_mort).ravel()

print("Final shape verification:")
print(f"  Xtr_liv : {Xtr_liv.shape}  ytr_liv : {ytr_liv.shape}")
print(f"  Xte_liv : {Xte_liv.shape}  yte_liv : {yte_liv.shape}")
print(f"  Xtr_mort: {Xtr_mort.shape} ytr_mort: {ytr_mort.shape}")
print(f"  Xte_mort: {Xte_mort.shape} yte_mort: {yte_mort.shape}")
print()

# ── Train Liver Disease ──
models_liv  = get_models()
results_liv, trained_liv = train_evaluate_clean(
    models_liv,
    Xtr_liv, ytr_liv,
    Xte_liv, yte_liv,
    "LIVER DISEASE PREDICTION"
)

print("\n")
print("="*90)
print("  LIVER DISEASE — MODEL COMPARISON")
print("="*90)
print(results_liv.to_string())

# ── Train Mortality ──
models_mort = get_models()
results_mort, trained_mort = train_evaluate_clean(
    models_mort,
    Xtr_mort, ytr_mort,
    Xte_mort, yte_mort,
    "MORTALITY PREDICTION"
)

print("\n")
print("="*90)
print("  MORTALITY — MODEL COMPARISON")
print("="*90)
print(results_mort.to_string())

best_liv_name  = results_liv.iloc[0]['Model']
best_mort_name = results_mort.iloc[0]['Model']
print(f"\n  Best Liver Disease model : {best_liv_name}")
print(f"  Best Mortality model     : {best_mort_name}")

In [ ]:
# ============================================================
# STEP 6 CELL 5 — ROC CURVES
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('ROC Curves — Trained on SMOTE | Tested on Original Data',
             fontsize=14, fontweight='bold')

colors_roc  = plt.cm.tab10(np.linspace(0, 1, 8))
line_styles = ['-', '--', '-.', ':', '-', '--', '-.', ':']

for ax, trained, X_te, y_te, title in [
    (axes[0], trained_liv,  X_te_liv_sc,  y_te_liv,  'Liver Disease'),
    (axes[1], trained_mort, X_te_mort_sc, y_te_mort, 'Mortality'),
]:
    for i, (name, model) in enumerate(trained.items()):
        y_proba = model.predict_proba(X_te)[:, 1]
        fpr, tpr, _ = roc_curve(y_te, y_proba)
        auc = roc_auc_score(y_te, y_proba)
        ax.plot(fpr, tpr, color=colors_roc[i],
                linestyle=line_styles[i], linewidth=2,
                label=f'{name} (AUC={auc:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.4, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.legend(loc='lower right', fontsize=8.5)
    ax.grid(alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('roc_curves_correct.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# STEP 6 CELL 6 — CONFUSION MATRICES + CLASSIFICATION REPORTS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Confusion Matrices — Best Models\n'
             f'Liver: {best_liv_name} | Mortality: {best_mort_name}',
             fontsize=13, fontweight='bold')

for ax, trained, X_te, y_te, best_name, title, labels in [
    (axes[0], trained_liv,  X_te_liv_sc,  y_te_liv,
     best_liv_name,  'Liver Disease', ['No Liver Dis.', 'Liver Dis.']),
    (axes[1], trained_mort, X_te_mort_sc, y_te_mort,
     best_mort_name, 'Mortality',     ['Survived', 'Died']),
]:
    y_pred = trained[best_name].predict(X_te)
    cm     = confusion_matrix(y_te, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm, annot=False, cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, cbar=False)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j + 0.5, i + 0.38, f'{cm[i,j]}',
                    ha='center', va='center', fontsize=18, fontweight='bold',
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
            ax.text(j + 0.5, i + 0.65, f'({cm_pct[i,j]:.1f}%)',
                    ha='center', va='center', fontsize=11,
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
    ax.set_title(f'{best_name}\n({title})', fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrices_correct.png', bbox_inches='tight', dpi=150)
plt.show()

# ── Classification Reports ──
for trained, X_te, y_te, best_name, title, labels in [
    (trained_liv,  X_te_liv_sc,  y_te_liv,
     best_liv_name,  'LIVER DISEASE', ['No Liver Disease', 'Liver Disease']),
    (trained_mort, X_te_mort_sc, y_te_mort,
     best_mort_name, 'MORTALITY',     ['Survived', 'Died']),
]:
    y_pred = trained[best_name].predict(X_te)
    print("=" * 60)
    print(f"  {title} — Best Model: {best_name}")
    print("=" * 60)
    print(classification_report(y_te, y_pred, target_names=labels))

In [ ]:
# ============================================================
# CLASSIFICATION REPORTS — Best Models
# ============================================================

for trained, X_te, y_te, best_name, title, labels in [
    (trained_liv,  X_test_liv,  y_test_liv,  best_liv_name,
     'LIVER DISEASE', ['No Liver Disease', 'Liver Disease']),
    (trained_mort, X_test_mort, y_test_mort, best_mort_name,
     'MORTALITY',     ['Survived', 'Died']),
]:
    y_pred = trained[best_name].predict(X_te)
    print("=" * 60)
    print(f"  {title} — Best Model: {best_name}")
    print("=" * 60)
    print(classification_report(y_te, y_pred, target_names=labels))

# ── Full comparison table ──
print("\n")
print("=" * 90)
print("  FULL MODEL COMPARISON — LIVER DISEASE (sorted by ROC-AUC)")
print("=" * 90)
print(results_liv.to_string())

print("\n")
print("=" * 90)
print("  FULL MODEL COMPARISON — MORTALITY (sorted by ROC-AUC)")
print("=" * 90)
print(results_mort.to_string())

In [ ]:
# ============================================================
# CROSS VALIDATION SUMMARY — 5-Fold Stratified
# ============================================================

print("=" * 65)
print("  5-FOLD CROSS VALIDATION SUMMARY")
print("=" * 65)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('5-Fold Cross Validation ROC-AUC Scores', fontsize=14, fontweight='bold')

for ax, results_df, trained, X_tr, y_tr, title in [
    (axes[0], results_liv,  trained_liv,  X_train_liv,  y_train_liv,  'Liver Disease'),
    (axes[1], results_mort, trained_mort, X_train_mort, y_train_mort, 'Mortality'),
]:
    cv_means = results_df.set_index('Model')['CV-AUC Mean']
    cv_stds  = results_df.set_index('Model')['CV-AUC Std']

    df_cv = pd.DataFrame({
        'Model'   : results_df['Model'],
        'CV_Mean' : results_df['CV-AUC Mean'],
        'CV_Std'  : results_df['CV-AUC Std'],
    }).sort_values('CV_Mean', ascending=True)

    colors_cv = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(df_cv)))
    bars = ax.barh(df_cv['Model'], df_cv['CV_Mean'],
                   xerr=df_cv['CV_Std'], color=colors_cv,
                   edgecolor='white', height=0.6,
                   error_kw=dict(elinewidth=1.5, capsize=4, ecolor='#333'))
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1, label='Random baseline')
    ax.set_title(f'{title}', fontweight='bold')
    ax.set_xlabel('CV ROC-AUC (Mean ± Std)')
    ax.set_xlim(0.4, 1.05)
    ax.legend(fontsize=9)
    for bar, mean, std in zip(bars, df_cv['CV_Mean'], df_cv['CV_Std']):
        ax.text(bar.get_width() + std + 0.005,
                bar.get_y() + bar.get_height() / 2,
                f'{mean:.3f}±{std:.3f}', va='center', fontsize=8)

    print(f"\n  {title}:")
    for _, row in results_df.iterrows():
        print(f"    {row['Model']:<22} CV-AUC: {row['CV-AUC Mean']:.4f} ± {row['CV-AUC Std']:.4f}")

plt.tight_layout()
plt.savefig('cv_comparison.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ============================================================
# CRITICAL VALIDATION: Test on ORIGINAL (non-SMOTE) data
# This is the REAL-WORLD performance check
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Build original (pre-SMOTE) test sets ──
# Use StandardScaler fit on original training data
from sklearn.model_selection import train_test_split

X_orig = liver_model[SELECTED_FEATURES].copy()
y_orig_liv  = liver_model['liver_disease'].copy()
y_orig_mort = liver_model['In-hospital_death'].copy()

# Same random_state=42 → reproducible original split
X_tr_orig_liv,  X_te_orig_liv,  y_tr_orig_liv,  y_te_orig_liv  = train_test_split(
    X_orig, y_orig_liv,  test_size=0.2, stratify=y_orig_liv,  random_state=42)

X_tr_orig_mort, X_te_orig_mort, y_tr_orig_mort, y_te_orig_mort = train_test_split(
    X_orig, y_orig_mort, test_size=0.2, stratify=y_orig_mort, random_state=42)

print("Original (non-SMOTE) test sets:")
print(f"  Liver  test: {X_te_orig_liv.shape}  | class dist: {dict(y_te_orig_liv.value_counts().sort_index())}")
print(f"  Mortality test: {X_te_orig_mort.shape} | class dist: {dict(y_te_orig_mort.value_counts().sort_index())}")
print(f"  Mortality rate in original test: {y_te_orig_mort.mean()*100:.1f}%  ← real-world imbalance\n")

# ── Evaluate SMOTE-trained models on original test data ──
def eval_on_original(trained_models, X_test_orig, y_test_orig, task_name, labels):
    results = []
    for name, pipe in trained_models.items():
        y_pred  = pipe.predict(X_test_orig)
        y_proba = pipe.predict_proba(X_test_orig)[:, 1]
        results.append({
            'Model'     : name,
            'Accuracy'  : round(accuracy_score(y_test_orig, y_pred), 4),
            'Precision' : round(precision_score(y_test_orig, y_pred, zero_division=0), 4),
            'Recall'    : round(recall_score(y_test_orig, y_pred, zero_division=0), 4),
            'F1-Score'  : round(f1_score(y_test_orig, y_pred, zero_division=0), 4),
            'ROC-AUC'   : round(roc_auc_score(y_test_orig, y_proba), 4),
            'PR-AUC'    : round(average_precision_score(y_test_orig, y_proba), 4),
        })
    df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
    df.index += 1

    print(f"{'='*80}")
    print(f"  {task_name} — Performance on ORIGINAL (non-SMOTE) test data")
    print(f"  (Real-world imbalanced distribution)")
    print(f"{'='*80}")
    print(df.to_string())
    print()
    return df

orig_results_liv  = eval_on_original(
    trained_liv,  X_te_orig_liv,  y_te_orig_liv,
    "LIVER DISEASE", ['No Liver Disease', 'Liver Disease']
)
orig_results_mort = eval_on_original(
    trained_mort, X_te_orig_mort, y_te_orig_mort,
    "MORTALITY", ['Survived', 'Died']
)

STEP 8: SHAP analysis

In [ ]:
# ============================================================
# STEP 8 — SHAP ANALYSIS
# Cell 8.1: Install & Import
# ============================================================

!pip install shap -q

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Feature names used during training (13 selected features)
# Update this list if your feature selection used different columns
feature_names = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin',
                 'Creatinine', 'BUN', 'Na', 'K', 'HCO3',
                 'Platelets', 'HR', 'Age']

print("✅ SHAP imported successfully")
print(f"   SHAP version: {shap.__version__}")
print(f"   Features ({len(feature_names)}): {feature_names}")

In [ ]:
# ============================================================
# STEP 8 — Cell 8.2: SHAP — Liver Disease (XGBoost)
# ============================================================

print("=" * 60)
print("  SHAP ANALYSIS — LIVER DISEASE (XGBoost)")
print("=" * 60)

# ── Build explainer on test set (original scale) ──
best_liv_model = trained_liv[best_liv_name]   # XGBoost

explainer_liv = shap.TreeExplainer(best_liv_model)
shap_values_liv = explainer_liv.shap_values(Xte_liv)

# If shap_values is list (multi-output), take class-1 slice
if isinstance(shap_values_liv, list):
    sv_liv = shap_values_liv[1]
else:
    sv_liv = shap_values_liv

print(f"  Test set shape     : {Xte_liv.shape}")
print(f"  SHAP values shape  : {sv_liv.shape}")
print(f"  Expected value (E[f(x)]): {explainer_liv.expected_value:.4f}"
      if not isinstance(explainer_liv.expected_value, list)
      else f"  Expected value class-1: {explainer_liv.expected_value[1]:.4f}")

# ── Convert test array back to DataFrame for SHAP plots ──
Xte_liv_df = pd.DataFrame(Xte_liv, columns=feature_names)

# ── 1. Beeswarm (Summary) Plot ──
plt.figure(figsize=(10, 6))
shap.summary_plot(
    sv_liv, Xte_liv_df,
    plot_type="dot",
    show=False,
    max_display=13
)
plt.title("SHAP Beeswarm — Liver Disease (XGBoost)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("shap_liv_beeswarm.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STEP 8 — Cell 8.3: SHAP Bar + Violin — Liver Disease
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Bar plot (mean |SHAP|) ──
plt.sca(axes[0])
shap.summary_plot(
    sv_liv, Xte_liv_df,
    plot_type="bar",
    show=False,
    max_display=13
)
axes[0].set_title("Mean |SHAP| — Liver Disease", fontsize=12, fontweight='bold')

# ── Violin plot ──
plt.sca(axes[1])
shap.summary_plot(
    sv_liv, Xte_liv_df,
    plot_type="violin",
    show=False,
    max_display=13
)
axes[1].set_title("SHAP Violin — Liver Disease", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig("shap_liv_bar_violin.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Ranked importance table ──
mean_shap_liv = pd.DataFrame({
    'Feature'    : feature_names,
    'Mean |SHAP|': np.abs(sv_liv).mean(axis=0)
}).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)
mean_shap_liv.index += 1

print("\n  RANKED LIVER BIOMARKERS — SHAP Importance")
print("  " + "="*40)
print(mean_shap_liv.to_string())

In [ ]:
# ============================================================
# STEP 8 — Cell 8.4: Waterfall & Force Plots — Liver Disease
# ============================================================

# ── Pick one high-risk and one low-risk sample ──
high_risk_idx = np.where(yte_liv == 1)[0][0]   # first actual liver disease case
low_risk_idx  = np.where(yte_liv == 0)[0][0]   # first non-liver disease case

ev_liv = (explainer_liv.expected_value[1]
          if isinstance(explainer_liv.expected_value, list)
          else explainer_liv.expected_value)

# ── Waterfall — High Risk ──
print("  Waterfall — High Risk (Liver Disease) Sample")
shap_exp_high = shap.Explanation(
    values    = sv_liv[high_risk_idx],
    base_values = ev_liv,
    data      = Xte_liv_df.iloc[high_risk_idx].values,
    feature_names = feature_names
)
plt.figure(figsize=(10, 5))
shap.waterfall_plot(shap_exp_high, show=False)
plt.title("Waterfall — Liver Disease (High Risk)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("shap_liv_waterfall_high.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Waterfall — Low Risk ──
print("  Waterfall — Low Risk (No Liver Disease) Sample")
shap_exp_low = shap.Explanation(
    values      = sv_liv[low_risk_idx],
    base_values = ev_liv,
    data        = Xte_liv_df.iloc[low_risk_idx].values,
    feature_names = feature_names
)
plt.figure(figsize=(10, 5))
shap.waterfall_plot(shap_exp_low, show=False)
plt.title("Waterfall — No Liver Disease (Low Risk)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("shap_liv_waterfall_low.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Force Plot (HTML) — High Risk ──
print("\n  Force Plot — High Risk Sample")
shap.initjs()
force_liv = shap.force_plot(
    ev_liv,
    sv_liv[high_risk_idx],
    Xte_liv_df.iloc[high_risk_idx],
    matplotlib=True,
    show=False
)
plt.title("Force Plot — Liver Disease High Risk", fontsize=11)
plt.tight_layout()
plt.savefig("shap_liv_force_high.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# STEP 8 — Cell 8.5: Dependence Plots — Liver Disease
# ============================================================

# Top 4 features from Cell 8.3
top4_liv = mean_shap_liv['Feature'].head(4).tolist()
print(f"  Top-4 features: {top4_liv}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top4_liv):
    plt.sca(axes[i])
    shap.dependence_plot(
        feat,
        sv_liv,
        Xte_liv_df,
        interaction_index="auto",
        ax=axes[i],
        show=False
    )
    axes[i].set_title(f"Dependence: {feat} → Liver Disease", fontsize=11, fontweight='bold')

plt.suptitle("SHAP Dependence Plots — Liver Disease (XGBoost)",
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("shap_liv_dependence.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STEP 8 — Cell 8.6: SHAP — Mortality (SVM → KernelExplainer)
# NOTE: SVM has no TreeExplainer; use KernelExplainer on
#       a small background sample (faster)
# ============================================================

print("=" * 60)
print("  SHAP ANALYSIS — MORTALITY (SVM — KernelExplainer)")
print("=" * 60)
print("  Building KernelExplainer (may take 1-2 min)...")

best_mort_model = trained_mort[best_mort_name]  # SVM

# Background: use 100-sample summary of training data (speeds up kernel SHAP)
background = shap.sample(Xtr_mort, 100, random_state=42)

explainer_mort = shap.KernelExplainer(
    best_mort_model.predict_proba,
    background
)

# Explain first 150 test samples (full 350 is slow for kernel SHAP)
print("  Computing SHAP values for 150 test samples...")
shap_values_mort = explainer_mort.shap_values(Xte_mort[:150], nsamples=200)

# class-1 (Died) SHAP values
if isinstance(shap_values_mort, list):
    sv_mort = shap_values_mort[1]
else: # assuming 3D array for binary classification (samples, features, classes)
    sv_mort = shap_values_mort[:, :, 1] # Select SHAP values for the positive class (index 1)

Xte_mort_df = pd.DataFrame(Xte_mort[:150], columns=feature_names)

print(f"  SHAP values shape  : {sv_mort.shape}")

In [ ]:
# ============================================================
# STEP 8 — Cell 8.7: SHAP Summary & Bar — Mortality
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Beeswarm ──
plt.sca(axes[0])
shap.summary_plot(
    sv_mort, Xte_mort_df,
    plot_type="dot",
    show=False,
    max_display=13
)
axes[0].set_title("SHAP Beeswarm — Mortality (SVM)", fontsize=12, fontweight='bold')

# ── Bar ──
plt.sca(axes[1])
shap.summary_plot(
    sv_mort, Xte_mort_df,
    plot_type="bar",
    show=False,
    max_display=13
)
axes[1].set_title("Mean |SHAP| — Mortality (SVM)", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig("shap_mort_summary.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Ranked importance ──
mean_shap_mort = pd.DataFrame({
    'Feature'    : feature_names,
    'Mean |SHAP|': np.abs(sv_mort).mean(axis=0)
}).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)
mean_shap_mort.index += 1

print("\n  RANKED BIOMARKERS — MORTALITY SHAP Importance")
print("  " + "="*40)
print(mean_shap_mort.to_string())

In [ ]:
# ============================================================
# STEP 8 — Cell 8.8: Permutation Importance — Both Tasks
# ============================================================

from sklearn.inspection import permutation_importance

print("  Computing permutation importance...")

# ── Liver ──
perm_liv = permutation_importance(
    best_liv_model, Xte_liv, yte_liv,
    n_repeats=20, random_state=42,
    scoring='roc_auc', n_jobs=-1
)
perm_liv_df = pd.DataFrame({
    'Feature'        : feature_names,
    'Importance Mean': perm_liv.importances_mean,
    'Importance Std' : perm_liv.importances_std
}).sort_values('Importance Mean', ascending=False).reset_index(drop=True)
perm_liv_df.index += 1

# ── Mortality ──
perm_mort = permutation_importance(
    best_mort_model, Xte_mort, yte_mort,
    n_repeats=20, random_state=42,
    scoring='roc_auc', n_jobs=-1
)
perm_mort_df = pd.DataFrame({
    'Feature'        : feature_names,
    'Importance Mean': perm_mort.importances_mean,
    'Importance Std' : perm_mort.importances_std
}).sort_values('Importance Mean', ascending=False).reset_index(drop=True)
perm_mort_df.index += 1

# ── Side-by-side plot ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df, title, color in zip(
    axes,
    [perm_liv_df, perm_mort_df],
    ['Liver Disease (XGBoost)', 'Mortality (SVM)'],
    ['steelblue', 'tomato']
):
    ax.barh(
        df['Feature'][::-1],
        df['Importance Mean'][::-1],
        xerr=df['Importance Std'][::-1],
        color=color, alpha=0.82,
        capsize=4, edgecolor='grey', linewidth=0.5
    )
    ax.set_xlabel("Mean ROC-AUC Decrease", fontsize=11)
    ax.set_title(f"Permutation Importance\n{title}", fontsize=12, fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig("shap_permutation_importance.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n  PERMUTATION IMPORTANCE — LIVER")
print(perm_liv_df.to_string())
print("\n  PERMUTATION IMPORTANCE — MORTALITY")
print(perm_mort_df.to_string())

In [ ]:
# ============================================================
# STEP 8 — Cell 8.9: Final Biomarker Importance Dashboard
# Combines SHAP + Permutation for both tasks
# ============================================================

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel 1: SHAP — Liver ──
ax1 = fig.add_subplot(gs[0, 0])
colors_liv = ['#2ecc71' if v > 0 else '#e74c3c'
              for v in mean_shap_liv['Mean |SHAP|']]
ax1.barh(
    mean_shap_liv['Feature'][::-1],
    mean_shap_liv['Mean |SHAP|'][::-1],
    color='#2ecc71', alpha=0.82, edgecolor='grey', linewidth=0.5
)
ax1.set_title("SHAP Importance\nLiver Disease (XGBoost)", fontweight='bold', fontsize=11)
ax1.set_xlabel("Mean |SHAP value|")
ax1.grid(axis='x', alpha=0.3)

# ── Panel 2: SHAP — Mortality ──
ax2 = fig.add_subplot(gs[0, 1])
ax2.barh(
    mean_shap_mort['Feature'][::-1],
    mean_shap_mort['Mean |SHAP|'][::-1],
    color='#e74c3c', alpha=0.82, edgecolor='grey', linewidth=0.5
)
ax2.set_title("SHAP Importance\nMortality (SVM)", fontweight='bold', fontsize=11)
ax2.set_xlabel("Mean |SHAP value|")
ax2.grid(axis='x', alpha=0.3)

# ── Panel 3: Permutation — Liver ──
ax3 = fig.add_subplot(gs[1, 0])
ax3.barh(
    perm_liv_df['Feature'][::-1],
    perm_liv_df['Importance Mean'][::-1],
    xerr=perm_liv_df['Importance Std'][::-1],
    color='#3498db', alpha=0.82, capsize=3,
    edgecolor='grey', linewidth=0.5
)
ax3.set_title("Permutation Importance\nLiver Disease (XGBoost)", fontweight='bold', fontsize=11)
ax3.set_xlabel("ROC-AUC Decrease")
ax3.axvline(0, color='black', linewidth=0.7, linestyle='--')
ax3.grid(axis='x', alpha=0.3)

# ── Panel 4: Permutation — Mortality ──
ax4 = fig.add_subplot(gs[1, 1])
ax4.barh(
    perm_mort_df['Feature'][::-1],
    perm_mort_df['Importance Mean'][::-1],
    xerr=perm_mort_df['Importance Std'][::-1],
    color='#e67e22', alpha=0.82, capsize=3,
    edgecolor='grey', linewidth=0.5
)
ax4.set_title("Permutation Importance\nMortality (SVM)", fontweight='bold', fontsize=11)
ax4.set_xlabel("ROC-AUC Decrease")
ax4.axvline(0, color='black', linewidth=0.7, linestyle='--')
ax4.grid(axis='x', alpha=0.3)

fig.suptitle("Biomarker Importance Dashboard — Liver Disease & Mortality",
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig("shap_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ STEP 8 — SHAP ANALYSIS COMPLETE")
print("   Saved: shap_liv_beeswarm, shap_liv_bar_violin,")
print("          shap_liv_waterfall_high/low, shap_liv_force_high,")
print("          shap_liv_dependence, shap_mort_summary,")
print("          shap_permutation_importance, shap_dashboard")

STEP 9: bayesian

In [ ]:
# !pip install pgmpy -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BIC
from pgmpy.inference import VariableElimination

print("✅ pgmpy imported successfully")

import pgmpy
print(f"pgmpy version: {pgmpy.__version__}")

In [ ]:
# ============================================================
# STEP 9 — Cell 9.2: Prepare & Discretize Data
# Bayesian Networks require discrete variables
# ============================================================

# ── Work from original liver_slim ──
bn_df = liver_slim.copy()

# ── Liver Disease label (reuse from your pipeline) ──
# If you already have y_liv as a series/array aligned to liver_slim, use it.
# Otherwise recreate using the same threshold logic you used earlier.
# We'll attach both targets:
bn_df['LiverDisease']      = y_liver_final.values   if hasattr(y_liver_final, 'values') else np.array(y_liver_final)
bn_df['In_hospital_death'] = bn_df['In-hospital_death'].astype(int)

# ── Select clinically meaningful features ──
bio_cols = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin',
            'Creatinine', 'BUN', 'Na', 'K', 'HCO3',
            'Platelets', 'HR', 'Age']

# ── Discretize into clinically meaningful bins ──
def discretize_bn(df):
    d = pd.DataFrame()

    d['Bilirubin']   = pd.cut(df['Bilirubin'],
                               bins=[-np.inf, 1.2, 3.0, np.inf],
                               labels=['Normal','Elevated','High'])

    d['ALT']         = pd.cut(df['ALT'],
                               bins=[-np.inf, 40, 120, np.inf],
                               labels=['Normal','Elevated','High'])

    d['AST']         = pd.cut(df['AST'],
                               bins=[-np.inf, 40, 120, np.inf],
                               labels=['Normal','Elevated','High'])

    d['ALP']         = pd.cut(df['ALP'],
                               bins=[-np.inf, 120, 300, np.inf],
                               labels=['Normal','Elevated','High'])

    d['Albumin']     = pd.cut(df['Albumin'],
                               bins=[-np.inf, 2.5, 3.5, np.inf],
                               labels=['Low','Borderline','Normal'])

    d['Creatinine']  = pd.cut(df['Creatinine'],
                               bins=[-np.inf, 1.2, 2.5, np.inf],
                               labels=['Normal','Elevated','High'])

    d['BUN']         = pd.cut(df['BUN'],
                               bins=[-np.inf, 20, 40, np.inf],
                               labels=['Normal','Elevated','High'])

    d['Na']          = pd.cut(df['Na'],
                               bins=[-np.inf, 135, 145, np.inf],
                               labels=['Low','Normal','High'])

    d['K']           = pd.cut(df['K'],
                               bins=[-np.inf, 3.5, 5.0, np.inf],
                               labels=['Low','Normal','High'])

    d['HCO3']        = pd.cut(df['HCO3'],
                               bins=[-np.inf, 22, 29, np.inf],
                               labels=['Low','Normal','High'])

    d['Platelets']   = pd.cut(df['Platelets'],
                               bins=[-np.inf, 100, 300, np.inf],
                               labels=['Low','Normal','High'])

    d['HR']          = pd.cut(df['HR'],
                               bins=[-np.inf, 60, 100, np.inf],
                               labels=['Low','Normal','High'])

    d['Age']         = pd.cut(df['Age'],
                               bins=[-np.inf, 45, 65, np.inf],
                               labels=['Young','Middle','Senior'])

    d['LiverDisease']      = df['LiverDisease'].astype(int).astype(str)
    d['In_hospital_death'] = df['In_hospital_death'].astype(int).astype(str)

    return d

bn_data = discretize_bn(bn_df)

# ── Drop any NaN rows created by binning ──
bn_data.dropna(inplace=True)
bn_data = bn_data.astype(str)

print(f"✅ Discretized dataset ready")
print(f"   Shape : {bn_data.shape}")
print(f"   Cols  : {list(bn_data.columns)}")
print(f"\n   LiverDisease distribution:\n{bn_data['LiverDisease'].value_counts()}")
print(f"\n   Mortality distribution:\n{bn_data['In_hospital_death'].value_counts()}")

In [ ]:
# ============================================================
# STEP 9 — Cell 9.3: Structure Learning via HillClimbSearch
# ============================================================

print("  Learning Bayesian Network structure...")
print("  (HillClimbSearch + BIC Score — may take ~30 sec)")

hc     = HillClimbSearch(bn_data)
best_model_struct = hc.estimate(
    scoring_method=BIC(bn_data),
    max_indegree=3,
    max_iter=500,
    show_progress=False
)

learned_edges = list(best_model_struct.edges())
print(f"\n✅ Structure learned — {len(learned_edges)} edges found:")
for e in learned_edges:
    print(f"   {e[0]}  →  {e[1]}")

In [ ]:
# ============================================================
# STEP 9 — Cell 9.4: Build + Fit Bayesian Network
# Safe DAG Construction
# ============================================================

# ── Expert/clinical edges ──
expert_edges = [

    # Liver biomarkers → latent liver disease
    ('Bilirubin',  'LiverDisease'),
    ('ALT',        'LiverDisease'),
    ('AST',        'LiverDisease'),
    ('ALP',        'LiverDisease'),
    ('Albumin',    'LiverDisease'),

    # Kidney-liver relationship
    ('Creatinine', 'BUN'),
    ('BUN',        'LiverDisease'),

    # Electrolytes
    ('Na',         'HCO3'),
    ('K',          'HCO3'),

    # Liver disease → mortality
    ('LiverDisease', 'In_hospital_death'),

    # Direct mortality influences
    ('Bilirubin',  'In_hospital_death'),
    ('Albumin',    'In_hospital_death'),
    ('Creatinine', 'In_hospital_death'),
    ('Na',         'In_hospital_death'),
    ('Platelets',  'In_hospital_death'),
    ('HR',         'In_hospital_death'),
    ('Age',        'In_hospital_death'),
]

# ============================================================
# REMOVE CONFLICTING LEARNED EDGES
# ============================================================

safe_learned_edges = []

for edge in learned_edges:

    parent, child = edge

    # Skip reverse/conflicting edges
    if (child, parent) in expert_edges:
        continue

    # Skip self-loops
    if parent == child:
        continue

    safe_learned_edges.append(edge)

# ============================================================
# MERGE EDGES
# ============================================================

all_edges = list(set(safe_learned_edges + expert_edges))

print(f"Original learned edges : {len(learned_edges)}")
print(f"Safe learned edges     : {len(safe_learned_edges)}")
print(f"Expert edges           : {len(expert_edges)}")
print(f"Merged (union) edges   : {len(all_edges)}")

In [ ]:
from pgmpy.estimators import MaximumLikelihoodEstimator

bn_model = BayesianNetwork(all_edges)

# ============================================================
# FIT MODEL CORRECTLY
# ============================================================

print("\nFitting Bayesian Network...")

# Instantiate the correct discrete estimator
# mle = MaximumLikelihoodEstimator(model=bn_model, data=bn_data)

# Fit CPDs
bn_model.fit(
    data=bn_data
    # estimator=mle # Removed explicit estimator, pgmpy will use DiscreteMLE() by default for discrete models
)

print("\n✅ Bayesian Network fitted successfully")

print(f"Nodes : {len(bn_model.nodes())}")
print(f"Edges : {len(bn_model.edges())}")

# ============================================================
# VERIFY CPDs
# ============================================================

print("\nChecking CPDs...")

for cpd in bn_model.get_cpds():
    print(f"✅ CPD learned for: {cpd.variable}")

print(f"\nTotal CPDs learned: {len(bn_model.get_cpds())}")

In [ ]:
# ============================================================
# STEP 9 — Cell 9.5: Visualize the Bayesian Network
# ============================================================

fig, ax = plt.subplots(figsize=(16, 10))

G = nx.DiGraph(bn_model.edges())

# ── Node color groups ──
liver_enzymes  = ['Bilirubin', 'ALT', 'AST', 'ALP', 'Albumin']
renal_electro  = ['Creatinine', 'BUN', 'Na', 'K', 'HCO3', 'Platelets']
clinical       = ['HR', 'Age']
targets        = ['LiverDisease', 'In_hospital_death']

color_map = []
for node in G.nodes():
    if node in targets:
        color_map.append('#e74c3c')       # red — target nodes
    elif node in liver_enzymes:
        color_map.append('#2ecc71')       # green — liver biomarkers
    elif node in renal_electro:
        color_map.append('#3498db')       # blue — renal/electrolytes
    else:
        color_map.append('#f39c12')       # orange — clinical

# ── Layout ──
pos = nx.spring_layout(G, seed=42, k=2.5)

nx.draw_networkx_nodes(G, pos, node_color=color_map,
                        node_size=2200, alpha=0.92, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8,
                         font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, edge_color='#555555',
                        arrows=True, arrowsize=20,
                        arrowstyle='->', width=1.5,
                        connectionstyle='arc3,rad=0.08',
                        ax=ax)

# ── Legend ──
legend_patches = [
    mpatches.Patch(color='#2ecc71', label='Liver Biomarkers'),
    mpatches.Patch(color='#3498db', label='Renal / Electrolytes'),
    mpatches.Patch(color='#f39c12', label='Clinical (HR, Age)'),
    mpatches.Patch(color='#e74c3c', label='Target Nodes'),
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=10)
ax.set_title("Bayesian Network — Liver Disease & Mortality Dependencies",
             fontsize=14, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig("bayesian_network_graph.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STEP 9 — Cell 9.6: Print Conditional Probability Tables
# ============================================================

print("=" * 65)
print("  CONDITIONAL PROBABILITY DISTRIBUTIONS (CPDs)")
print("=" * 65)

# ── LiverDisease CPD ──
print("\n CPD — LiverDisease")
print("-" * 50)
cpd_liv = bn_model.get_cpds('LiverDisease')
print(cpd_liv)

# ── Mortality CPD ──
print("\n CPD — In_hospital_death")
print("-" * 50)
cpd_mort = bn_model.get_cpds('In_hospital_death')
print(cpd_mort)

# ── Bilirubin CPD (root node) ──
print("\n CPD — Bilirubin (root biomarker)")
print("-" * 50)
cpd_bili = bn_model.get_cpds('Bilirubin')
print(cpd_bili)

# ── Albumin CPD ──
print("\n CPD — Albumin")
print("-" * 50)
cpd_alb = bn_model.get_cpds('Albumin')
print(cpd_alb)

In [ ]:
# ============================================================
# STEP 9 — Cell 9.7: Variable Elimination — Inference Queries
# ============================================================

infer = VariableElimination(bn_model)

print("=" * 65)
print("  BAYESIAN NETWORK INFERENCE — CONDITIONAL PROBABILITIES")
print("=" * 65)

# ────────────────────────────────────────────────
# Query 1: P(Mortality | Bilirubin=High, Albumin=Low)
# ────────────────────────────────────────────────
print("\n Query 1: P(Death | Bilirubin=High, Albumin=Low)")
q1 = infer.query(
    variables=['In_hospital_death'],
    evidence={'Bilirubin': 'High', 'Albumin': 'Low'},
    show_progress=False
)
print(q1)

# ────────────────────────────────────────────────
# Query 2: P(Mortality | Bilirubin=Normal, Albumin=Normal)
# ────────────────────────────────────────────────
print("\n Query 2: P(Death | Bilirubin=Normal, Albumin=Normal)")
q2 = infer.query(
    variables=['In_hospital_death'],
    evidence={'Bilirubin': 'Normal', 'Albumin': 'Normal'},
    show_progress=False
)
print(q2)

# ────────────────────────────────────────────────
# Query 3: P(LiverDisease | AST=High, ALT=High, Bilirubin=High)
# ────────────────────────────────────────────────
print("\n Query 3: P(LiverDisease | AST=High, ALT=High, Bilirubin=High)")
q3 = infer.query(
    variables=['LiverDisease'],
    evidence={'AST': 'High', 'ALT': 'High', 'Bilirubin': 'High'},
    show_progress=False
)
print(q3)

# ────────────────────────────────────────────────
# Query 4: P(LiverDisease | AST=Normal, ALT=Normal, Albumin=Normal)
# ────────────────────────────────────────────────
print("\n Query 4: P(LiverDisease | AST=Normal, ALT=Normal, Albumin=Normal)")
q4 = infer.query(
    variables=['LiverDisease'],
    evidence={'AST': 'Normal', 'ALT': 'Normal', 'Albumin': 'Normal'},
    show_progress=False
)
print(q4)

# ────────────────────────────────────────────────
# Query 5: P(Mortality | Age=Senior, Creatinine=High, Na=Low)
# ────────────────────────────────────────────────
print("\n Query 5: P(Death | Age=Senior, Creatinine=High, Na=Low)")
q5 = infer.query(
    variables=['In_hospital_death'],
    evidence={'Age': 'Senior', 'Creatinine': 'High', 'Na': 'Low'},
    show_progress=False
)
print(q5)

# ────────────────────────────────────────────────
# Query 6: P(Mortality | LiverDisease=1)
# ────────────────────────────────────────────────
print("\n Query 6: P(Death | LiverDisease=1 [confirmed])")
q6 = infer.query(
    variables=['In_hospital_death'],
    evidence={'LiverDisease': '1'},
    show_progress=False
)
print(q6)

print("\n Query 7: P(Death | LiverDisease=0 [no liver disease])")
q7 = infer.query(
    variables=['In_hospital_death'],
    evidence={'LiverDisease': '0'},
    show_progress=False
)
print(q7)

In [ ]:
# ============================================================
# STEP 9 — Cell 9.8: Mortality Risk under Different Scenarios
# ============================================================

scenarios = {
    'Healthy\n(Normal labs)'          : {'Bilirubin':'Normal','Albumin':'Normal','Creatinine':'Normal','Age':'Young'},
    'Mild Liver\nDisease'             : {'Bilirubin':'Elevated','Albumin':'Borderline','ALT':'Elevated'},
    'Severe Liver\nDisease'           : {'Bilirubin':'High','Albumin':'Low','AST':'High'},
    'Liver Disease\n+ Renal Failure'  : {'Bilirubin':'High','Albumin':'Low','Creatinine':'High','BUN':'High'},
    'Senior +\nMulti-organ'           : {'Age':'Senior','Creatinine':'High','Na':'Low','HR':'High'},
    'Confirmed\nLiver Disease'        : {'LiverDisease':'1'},
    'No Liver\nDisease'               : {'LiverDisease':'0'},
}

mortality_probs = {}
for label, evidence in scenarios.items():
    try:
        # filter evidence to only nodes that exist in the model
        valid_ev = {k: v for k, v in evidence.items()
                    if k in bn_model.nodes()}
        result = infer.query(
            variables=['In_hospital_death'],
            evidence=valid_ev,
            show_progress=False
        )
        # P(death=1)
        states = result.state_names['In_hospital_death']
        idx    = states.index('1')
        mortality_probs[label] = result.values[idx]
    except Exception as ex:
        print(f"  ⚠️  Skipped '{label}': {ex}")
        mortality_probs[label] = 0.0

# ── Plot ──
fig, ax = plt.subplots(figsize=(13, 6))

labels = list(mortality_probs.keys())
probs  = [mortality_probs[l] * 100 for l in labels]

bar_colors = ['#2ecc71' if p < 25 else '#f39c12' if p < 50 else '#e74c3c'
              for p in probs]

bars = ax.bar(labels, probs, color=bar_colors, edgecolor='grey',
              linewidth=0.7, alpha=0.88, width=0.6)

for bar, prob in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.8,
            f'{prob:.1f}%',
            ha='center', va='bottom',
            fontsize=10, fontweight='bold')

ax.axhline(19.1, color='navy', linewidth=1.5,
           linestyle='--', label='Overall mortality rate (19.1%)')

ax.set_ylabel("Probability of In-Hospital Death (%)", fontsize=11)
ax.set_title("Bayesian Network — Mortality Risk Under Clinical Scenarios",
             fontsize=13, fontweight='bold')
ax.set_ylim(0, min(max(probs) + 15, 100))
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("bn_mortality_scenarios.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STEP 9 — Cell 9.9: Markov Blankets + D-Separation Analysis
# ============================================================

print("=" * 60)
print("  MARKOV BLANKETS — Direct Influence Neighborhoods")
print("=" * 60)

for target in ['LiverDisease', 'In_hospital_death']:
    mb = bn_model.get_markov_blanket(target)
    print(f"\n  Markov Blanket of '{target}':")
    print(f"  → {sorted(mb)}")

print("\n")
print("=" * 60)
print("  D-SEPARATION TESTS")
print("=" * 60)

from pgmpy.independencies import Independencies

tests = [
    ('Bilirubin',  'In_hospital_death', ['LiverDisease']),
    ('ALT',        'In_hospital_death', ['LiverDisease']),
    ('Age',        'LiverDisease',      ['HR']),
    ('Creatinine', 'LiverDisease',      ['BUN']),
]

for x, y, z in tests:
    try:
        result = bn_model.active_trail_nodes(x, observed=z)
        separated = y not in result
        print(f"  d-sep({x} ⊥ {y} | {z}) = {separated}")
    except Exception as ex:
        print(f"  ⚠️  {x} ⊥ {y} | {z} — {ex}")

print("\n")
print("=" * 60)
print("  EDGE STRENGTH — Strength of Dependencies")
print("=" * 60)

try:
    from pgmpy.estimators import StructureScore
    print("  Top edges by influence on LiverDisease and Mortality:")
    parents_liv  = list(bn_model.get_parents('LiverDisease'))
    parents_mort = list(bn_model.get_parents('In_hospital_death'))
    print(f"  LiverDisease  parents : {parents_liv}")
    print(f"  Mortality     parents : {parents_mort}")
except Exception as ex:
    print(f"  {ex}")


STEP 10 — Mortality Risk Stratification (Low / Medium / High)

In [ ]:
# ============================================================
# STEP 10 — MORTALITY RISK STRATIFICATION
# Cell 10.1: Generate Predicted Probabilities
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

# ── Use best mortality model (SVM) on full test set ──
best_mort_model = trained_mort[best_mort_name]

# Predicted probabilities on original test set
y_mort_proba = best_mort_model.predict_proba(Xte_mort)[:, 1]

print(f"  Best Mortality Model  : {best_mort_name}")
print(f"  Test samples          : {len(y_mort_proba)}")
print(f"  Prob range            : [{y_mort_proba.min():.4f}, {y_mort_proba.max():.4f}]")
print(f"  Mean predicted prob   : {y_mort_proba.mean():.4f}")
print(f"  Actual mortality rate : {yte_mort.mean():.4f}")

In [ ]:
# ============================================================
# STEP 10 — Cell 10.2: Data-Driven Thresholds
# Based on predicted probability distribution percentiles
# ============================================================

import numpy as np
import pandas as pd

# ── Step 1: Examine probability distribution ──
print("=" * 55)
print("  PREDICTED PROBABILITY DISTRIBUTION ANALYSIS")
print("=" * 55)

percentiles = [10, 25, 33, 50, 67, 75, 90]
print(f"\n  Percentile analysis of predicted mortality probs:")
for p in percentiles:
    val = np.percentile(y_mort_proba, p)
    print(f"    P{p:>2}  =  {val:.4f}  ({val*100:.1f}%)")

print(f"\n  Mean  : {y_mort_proba.mean():.4f}")
print(f"  Std   : {y_mort_proba.std():.4f}")
print(f"  Min   : {y_mort_proba.min():.4f}")
print(f"  Max   : {y_mort_proba.max():.4f}")

# ── Step 2: Data-driven thresholds (33rd and 67th percentile) ──
LOW_THRESH  = np.percentile(y_mort_proba, 33)   # bottom third  = Low
HIGH_THRESH = np.percentile(y_mort_proba, 67)   # top third     = High

print(f"\n  DATA-DRIVEN THRESHOLDS (33rd / 67th percentile):")
print(f"    Low Risk    : Prob < {LOW_THRESH:.4f}  ({LOW_THRESH*100:.1f}%)")
print(f"    Medium Risk : {LOW_THRESH:.4f} ≤ Prob < {HIGH_THRESH:.4f}")
print(f"    High Risk   : Prob ≥ {HIGH_THRESH:.4f}  ({HIGH_THRESH*100:.1f}%)")

# ── Step 3: Assign risk tiers ──
def assign_risk(prob):
    if prob < LOW_THRESH:
        return 'Low Risk'
    elif prob < HIGH_THRESH:
        return 'Medium Risk'
    else:
        return 'High Risk'

risk_df = pd.DataFrame({
    'True_Label'     : yte_mort,
    'Mortality_Prob' : y_mort_proba,
    'Risk_Category'  : [assign_risk(p) for p in y_mort_proba]
})

risk_order  = ['Low Risk', 'Medium Risk', 'High Risk']
risk_counts = risk_df['Risk_Category'].value_counts().reindex(risk_order)

print(f"\n  {'Risk Category':<15} {'Count':>6} {'% Total':>9}")
print(f"  {'-'*35}")
for cat in risk_order:
    n   = risk_counts[cat]
    pct = n / len(risk_df) * 100
    print(f"  {cat:<15} {n:>6}   {pct:>7.1f}%")

print(f"\n  Total test samples : {len(risk_df)}")
print(f"\n  WHY THESE THRESHOLDS?")
print(f"  → Derived from the model's own predicted probability")
print(f"    distribution using 33rd and 67th percentiles.")
print(f"  → Splits patients into equal thirds: bottom 33% = Low,")
print(f"    middle 34% = Medium, top 33% = High.")
print(f"  → No arbitrary clinical cutoffs — fully data-driven.")
print(f"  → Actual mortality labels come from PhysioNet ICU")
print(f"    dataset (In-hospital_death), NOT from our rules.")

In [ ]:
# ============================================================
# STEP 10 — Cell 10.3: Actual Mortality Rate per Risk Tier
# ============================================================

print("=" * 60)
print("  ACTUAL MORTALITY RATE PER RISK CATEGORY")
print("=" * 60)

summary = risk_df.groupby('Risk_Category').agg(
    Count          = ('True_Label', 'count'),
    Deaths         = ('True_Label', 'sum'),
    Actual_Mort_Rate = ('True_Label', 'mean'),
    Mean_Pred_Prob = ('Mortality_Prob', 'mean'),
    Min_Prob       = ('Mortality_Prob', 'min'),
    Max_Prob       = ('Mortality_Prob', 'max')
).reindex(risk_order).round(4)

summary['Actual_Mort_%'] = (summary['Actual_Mort_Rate'] * 100).round(1)

print(f"\n{summary.to_string()}")

print("\n  Interpretation:")
for cat in risk_order:
    row = summary.loc[cat]
    print(f"   {cat:<12}: "
          f"{int(row['Count'])} patients | "
          f"Actual death rate = {row['Actual_Mort_%']}% | "
          f"Mean predicted prob = {row['Mean_Pred_Prob']:.3f}")

In [ ]:
# ============================================================
# STEP 10 — Cell 10.4: Risk Stratification Visualizations
# ============================================================

risk_colors = {
    'Low Risk'    : '#2ecc71',
    'Medium Risk' : '#f39c12',
    'High Risk'   : '#e74c3c'
}

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

# ── Plot 1: Risk distribution (pie) ──
ax1 = fig.add_subplot(gs[0, 0])
wedge_colors = [risk_colors[r] for r in risk_order]
ax1.pie(
    risk_counts.values,
    labels=risk_order,
    colors=wedge_colors,
    autopct='%1.1f%%',
    startangle=140,
    textprops={'fontsize': 10},
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
ax1.set_title("Risk Category Distribution", fontweight='bold', fontsize=11)

# ── Plot 2: Count bar chart ──
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.bar(
    risk_order,
    risk_counts.values,
    color=wedge_colors,
    edgecolor='grey', linewidth=0.6, alpha=0.88
)
for bar, val in zip(bars, risk_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 2,
             str(val), ha='center', fontsize=10, fontweight='bold')
ax2.set_ylabel("Number of Patients")
ax2.set_title("Patient Count by Risk Tier", fontweight='bold', fontsize=11)
ax2.grid(axis='y', alpha=0.3)

# ── Plot 3: Actual mortality rate per tier ──
ax3 = fig.add_subplot(gs[0, 2])
actual_rates = [summary.loc[r, 'Actual_Mort_%'] for r in risk_order]
bars3 = ax3.bar(
    risk_order, actual_rates,
    color=wedge_colors,
    edgecolor='grey', linewidth=0.6, alpha=0.88
)
for bar, val in zip(bars3, actual_rates):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax3.axhline(yte_mort.mean()*100, color='navy', linewidth=1.5,
            linestyle='--', label=f'Overall rate ({yte_mort.mean()*100:.1f}%)')
ax3.set_ylabel("Actual Mortality Rate (%)")
ax3.set_title("Actual Mortality Rate per Tier", fontweight='bold', fontsize=11)
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.3)

# ── Plot 4: Probability distribution by risk tier ──
ax4 = fig.add_subplot(gs[1, 0:2])
for cat in risk_order:
    subset = risk_df[risk_df['Risk_Category'] == cat]['Mortality_Prob']
    ax4.hist(subset, bins=25, alpha=0.65,
             color=risk_colors[cat], label=cat,
             edgecolor='white', linewidth=0.4)
ax4.axvline(LOW_THRESH,  color='#f39c12', linewidth=2,
            linestyle='--', label=f'Low/Med threshold ({LOW_THRESH:.0%})')
ax4.axvline(HIGH_THRESH, color='#e74c3c', linewidth=2,
            linestyle='--', label=f'Med/High threshold ({HIGH_THRESH:.0%})')
ax4.set_xlabel("Predicted Mortality Probability")
ax4.set_ylabel("Count")
ax4.set_title("Mortality Probability Distribution by Risk Tier",
              fontweight='bold', fontsize=11)
ax4.legend(fontsize=9)
ax4.grid(alpha=0.3)

# ── Plot 5: Violin — prob by true outcome ──
ax5 = fig.add_subplot(gs[1, 2])
for i, (label, color) in enumerate(zip(['Survived (0)', 'Died (1)'],
                                        ['#3498db', '#e74c3c'])):
    mask   = yte_mort == i
    data   = y_mort_proba[mask]
    parts  = ax5.violinplot(data, positions=[i], widths=0.6,
                             showmedians=True, showextrema=True)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
ax5.set_xticks([0, 1])
ax5.set_xticklabels(['Survived', 'Died'])
ax5.set_ylabel("Predicted Mortality Probability")
ax5.set_title("Predicted Prob by True Outcome",
              fontweight='bold', fontsize=11)
ax5.grid(axis='y', alpha=0.3)

fig.suptitle("STEP 10 — Mortality Risk Stratification Dashboard",
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig("mortality_risk_stratification.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STEP 10 — Cell 10.5: Detailed Classification Report
# ============================================================

print("=" * 65)
print("  CLASSIFICATION REPORT — RISK TIER vs ACTUAL OUTCOME")
print("=" * 65)

# ── Per-tier breakdown ──
for cat in risk_order:
    subset = risk_df[risk_df['Risk_Category'] == cat]
    total  = len(subset)
    died   = subset['True_Label'].sum()
    surv   = total - died
    rate   = died / total * 100 if total > 0 else 0

    print(f"\n  {'─'*50}")
    print(f"  {cat.upper()}")
    print(f"  {'─'*50}")
    print(f"  Total patients : {total}")
    print(f"  Survived       : {surv}  ({surv/total*100:.1f}%)")
    print(f"  Died           : {died}  ({rate:.1f}%)")
    print(f"  Mean Pred Prob : {subset['Mortality_Prob'].mean():.4f}")

# ── Binary classification using risk as predictor ──
# Map: Low=0, Medium=0, High=1  (High risk → predicted death)
risk_binary = (risk_df['Risk_Category'] == 'High Risk').astype(int)

print(f"\n\n{'='*65}")
print("  BINARY EVAL: High Risk = Predicted Death")
print(f"{'='*65}")
print(classification_report(
    yte_mort, risk_binary,
    target_names=['Survived', 'Died']
))

print(f"  ROC-AUC (prob-based) : "
      f"{roc_auc_score(yte_mort, y_mort_proba):.4f}")

In [ ]:
# ============================================================
# STEP 10 — Cell 10.6: Confusion Matrices
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Matrix 1: Original binary prediction ──
cm1 = confusion_matrix(yte_mort, best_mort_model.predict(Xte_mort))
sns.heatmap(
    cm1, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Survived', 'Died'],
    yticklabels=['Survived', 'Died'],
    ax=axes[0], linewidths=0.5
)
axes[0].set_title(f"Confusion Matrix\n{best_mort_name} (Binary)",
                  fontweight='bold', fontsize=11)
axes[0].set_ylabel("Actual")
axes[0].set_xlabel("Predicted")

# ── Matrix 2: Risk tier vs actual ──
risk_num = risk_df['Risk_Category'].map(
    {'Low Risk': 0, 'Medium Risk': 1, 'High Risk': 2}
)
actual_num = yte_mort.copy()

# Build 3-class confusion (actual: 0=survived,1=died vs tier: 0,1,2)
cm2_data = pd.crosstab(
    risk_df['True_Label'].map({0: 'Survived', 1: 'Died'}),
    risk_df['Risk_Category'],
    rownames=['Actual'],
    colnames=['Predicted Risk']
)[risk_order]

sns.heatmap(
    cm2_data, annot=True, fmt='d', cmap='RdYlGn_r',
    ax=axes[1], linewidths=0.5
)
axes[1].set_title("Risk Tier vs Actual Outcome",
                  fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig("mortality_confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STEP 10 — Cell 10.7: Reusable Risk Scoring Function
# ============================================================

def predict_mortality_risk(model, X, feature_names=None,
                            low_thresh=0.20, high_thresh=0.45):
    """
    Returns DataFrame with:
      - Mortality_Prob   : predicted probability of death
      - Risk_Category    : Low / Medium / High Risk
      - Risk_Score       : 1 / 2 / 3 (for sorting)
      - Confidence       : distance from nearest threshold
    """
    X = np.array(X)
    proba = model.predict_proba(X)[:, 1]

    categories, scores, confidence = [], [], []
    for p in proba:
        if p < low_thresh:
            categories.append('Low Risk')
            scores.append(1)
            confidence.append(low_thresh - p)
        elif p < high_thresh:
            categories.append('Medium Risk')
            scores.append(2)
            confidence.append(min(p - low_thresh, high_thresh - p))
        else:
            categories.append('High Risk')
            scores.append(3)
            confidence.append(p - high_thresh)

    result = pd.DataFrame({
        'Mortality_Prob'  : proba.round(4),
        'Risk_Category'   : categories,
        'Risk_Score'      : scores,
        'Confidence'      : [round(c, 4) for c in confidence]
    })

    if feature_names is not None:
        feat_df = pd.DataFrame(X, columns=feature_names)
        result  = pd.concat([feat_df.reset_index(drop=True),
                             result.reset_index(drop=True)], axis=1)
    return result


# ── Demo on test set ──
risk_result = predict_mortality_risk(
    best_mort_model, Xte_mort,
    feature_names=feature_names
)

print("  Sample predictions (first 10):")
print(risk_result[['Mortality_Prob','Risk_Category',
                    'Risk_Score','Confidence']].head(10).to_string())

print(f"\n  Risk distribution on full test set:")
print(risk_result['Risk_Category'].value_counts().reindex(risk_order))


STEP 11: Final Model + Save PKL Files

In [ ]:
# ============================================================
# STEP 11 — COMPLETE FIX CELL
# Rebuilds y_liv from liver_slim + saves everything correctly
# ============================================================

import joblib, os, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

os.makedirs('saved_models', exist_ok=True)

# ── Step 1: Rebuild y_liv from liver_slim ──
# Liver disease label: 1 if any liver enzyme is elevated
# (same logic your pipeline used — we recreate it from raw data)
print("  Step 1: Rebuilding y_liv from liver_slim...")

y_liv_raw  = (
    (liver_slim['Bilirubin']  > 1.2).astype(int) |
    (liver_slim['ALT']        > 40 ).astype(int) |
    (liver_slim['AST']        > 40 ).astype(int) |
    (liver_slim['ALP']        > 120).astype(int)
).astype(int).values

y_mort_raw = liver_slim['In-hospital_death'].astype(int).values

print(f"  y_liv  — shape: {y_liv_raw.shape} | "
      f"positive: {y_liv_raw.sum()} ({y_liv_raw.mean()*100:.1f}%)")
print(f"  y_mort — shape: {y_mort_raw.shape} | "
      f"positive: {y_mort_raw.sum()} ({y_mort_raw.mean()*100:.1f}%)")

# ── Step 2: Get raw features from liver_slim ──
print("\n  Step 2: Extracting raw features...")
raw_X = liver_slim[feature_names].values
print(f"  raw_X shape : {raw_X.shape}")
print(f"  raw_X range : min={raw_X.min():.2f}, max={raw_X.max():.2f}")

# ── Step 3: Fit fresh StandardScalers on raw data ──
print("\n  Step 3: Fitting fresh scalers on raw data...")
new_scaler_liv  = StandardScaler().fit(raw_X)
new_scaler_mort = StandardScaler().fit(raw_X)
print("  ✅ Scalers fitted")

# ── Step 4: Sanity check — Patient A (Healthy) ──
print("\n  Step 4: Sanity checks...")
pA = np.array([[0.8, 22, 20, 70, 4.2, 0.9,
                14, 140, 4.0, 25, 220, 72, 32]])

pA_sl = new_scaler_liv.transform(pA)
pA_sm = new_scaler_mort.transform(pA)

liv_A  = trained_liv[best_liv_name].predict_proba(pA_sl)[0][1]
mort_A = trained_mort[best_mort_name].predict_proba(pA_sm)[0][1]

print(f"  Patient A — Healthy:")
print(f"    Liver prob  : {liv_A*100:.2f}%  "
      f"{'✅ OK' if liv_A < 0.5 else '❌ Still wrong'}")
print(f"    Mort  prob  : {mort_A*100:.2f}%  "
      f"{'✅ OK' if mort_A < 0.25 else '⚠ Check'}")

# ── Sanity check — Patient C (Severe) ──
pC = np.array([[8.2, 320, 410, 520, 2.1, 3.8,
                65, 128, 5.8, 14, 55, 118, 74]])

pC_sl = new_scaler_liv.transform(pC)
pC_sm = new_scaler_mort.transform(pC)

liv_C  = trained_liv[best_liv_name].predict_proba(pC_sl)[0][1]
mort_C = trained_mort[best_mort_name].predict_proba(pC_sm)[0][1]

print(f"\n  Patient C — Severe Multi-Organ Failure:")
print(f"    Liver prob  : {liv_C*100:.2f}%  "
      f"{'✅ OK' if liv_C > 0.8 else '⚠ Check'}")
print(f"    Mort  prob  : {mort_C*100:.2f}%  "
      f"{'✅ OK' if mort_C > 0.25 else '⚠ Check'}")

# ── Patient B (Moderate) ──
pB = np.array([[3.5, 150, 130, 280, 3.0, 1.3,
                28, 136, 3.8, 21, 130, 88, 58]])

pB_sl = new_scaler_liv.transform(pB)
pB_sm = new_scaler_mort.transform(pB)

liv_B  = trained_liv[best_liv_name].predict_proba(pB_sl)[0][1]
mort_B = trained_mort[best_mort_name].predict_proba(pB_sm)[0][1]

print(f"\n  Patient B — Moderate Mortality Risk:")
print(f"    Liver prob  : {liv_B*100:.2f}%")
print(f"    Mort  prob  : {mort_B*100:.2f}%")

# ── Step 5: Save all artifacts ──
print("\n  Step 5: Saving artifacts...")

joblib.dump(trained_liv[best_liv_name],   'saved_models/liver_disease_model.pkl')
joblib.dump(trained_mort[best_mort_name], 'saved_models/mortality_model.pkl')
joblib.dump(new_scaler_liv,               'saved_models/scaler_liver.pkl')
joblib.dump(new_scaler_mort,              'saved_models/scaler_mortality.pkl')
joblib.dump(feature_names,                'saved_models/feature_names.pkl')
joblib.dump({
    'low_thresh'   : float(LOW_THRESH),
    'high_thresh'  : float(HIGH_THRESH),
    'feature_names': feature_names
}, 'saved_models/thresholds.pkl')

print(f"\n  {'File':<42} {'Size':>8}")
print(f"  {'-'*52}")
for f in sorted(os.listdir('saved_models')):
    size = os.path.getsize(f'saved_models/{f}') / 1024
    print(f"  {f:<42} {size:>7.1f} KB")

print("\n✅ ALL FIXED AND SAVED")
print(f"\n  Summary:")
print(f"    Liver model  : {best_liv_name}")
print(f"    Mort  model  : {best_mort_name}")
print(f"    Low thresh   : {LOW_THRESH:.4f} ({LOW_THRESH*100:.1f}%)")
print(f"    High thresh  : {HIGH_THRESH:.4f} ({HIGH_THRESH*100:.1f}%)")

In [ ]:
# ── Download all corrected pkl files ──
from google.colab import files
import os

print("  Downloading corrected model files...")
for fname in sorted(os.listdir('saved_models')):
    files.download(f'saved_models/{fname}')
    print(f"  ✅ {fname}")

print("\n  Done! Place all files in saved_models/ folder next to app.py")